# AML ML Preparation — EDA, Feature Selection & Baseline Model
---
**Target**: `is_aml` (binary)

**Feature Strategy:**
- **RETAIN AS-IS**: All 126 rule flags + encoded categoricals (no selection applied)
- **SELECT FROM**: Graph/velocity/balance features only (correlation + VIF + importance)
- **EXCLUDE**: Typology signal, convergence risk, temporal risk (leakage from typology labels)
- **EXCLUDE**: FIS, alert_level, fis_band (post-hoc derived scores)

**Class Imbalance**: SMOTE, class weights, and threshold tuning compared


## 1 — Environment Setup


In [1]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings("ignore")
from collections import defaultdict
from datetime import datetime

# Viz
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 6)

OUTPUT_DIR = "ml_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Environment ready")



Environment ready


## 2 — Load Feature-Engineered Data


In [2]:
INPUT_FILE = "../outputs_updated/stg_transactions_features_V2.parquet"

if not os.path.exists(INPUT_FILE):
    # Try alternate paths
    for alt in ["stg_transactions_features_V2.parquet",
                "../aml_features_output/stg_transactions_features_V2.parquet",
                "../stg_transactions_features_V2.parquet"]:
        if os.path.exists(alt):
            INPUT_FILE = alt
            break

df = pd.read_parquet(INPUT_FILE)
print(f"Loaded: {INPUT_FILE}")
print(f"  {len(df):,} rows × {len(df.columns)} columns")
print(f"  is_aml distribution: 0={( df['is_aml']==0).sum():,}  1={(df['is_aml']==1).sum():,}  ({(df['is_aml']==1).mean()*100:.1f}%)")



Loaded: ../outputs_updated/stg_transactions_features_V2.parquet
  333,875 rows × 318 columns
  is_aml distribution: 0=204,230  1=129,645  (38.8%)


## 3 — Initial EDA: Target Distribution & Data Quality


In [3]:
print("=" * 90)
print("INITIAL EDA")
print("=" * 90)

# ── 3.1: Target distribution ──
print("\n── 3.1: Target Variable (is_aml) ──")
target_counts = df["is_aml"].value_counts().sort_index()
for val, cnt in target_counts.items():
    print(f"  is_aml={val}: {cnt:>10,} ({cnt/len(df)*100:.1f}%)")
imbalance_ratio = target_counts[0] / max(target_counts[1], 1)
print(f"  Imbalance ratio: {imbalance_ratio:.1f}:1 (Clean:AML)")

# ── 3.2: Column type breakdown ──
print("\n── 3.2: Column Types ──")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
object_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
print(f"  Numeric: {len(numeric_cols)} | Object/String: {len(object_cols)} | Boolean: {len(bool_cols)}")

# ── 3.3: Missing values ──
print("\n── 3.3: Missing Values (top 20) ──")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).sort_values(ascending=False)
missing_top = missing_pct[missing_pct > 0].head(20)
if len(missing_top) > 0:
    for col, pct in missing_top.items():
        print(f"  {col:<50s} {pct:>6.2f}%")
else:
    print("  No missing values found")

# ── 3.4: Typology distribution within AML ──
print("\n── 3.4: Typology Distribution (within is_aml=1) ──")
if "aml_typology" in df.columns:
    aml_df = df[df["is_aml"] == 1]
    all_typs = {}
    for t in aml_df["aml_typology"].dropna():
        for part in str(t).split("; "):
            part = part.strip()
            if part: all_typs[part] = all_typs.get(part, 0) + 1
    for typ, cnt in sorted(all_typs.items(), key=lambda x: -x[1]):
        print(f"  {typ:<40s} {cnt:>8,} ({cnt/len(aml_df)*100:.1f}%)")

# ── 3.5: Key numeric feature statistics ──
print("\n── 3.5: Key Feature Statistics (AML vs Clean) ──")
key_features = ["transaction_amount", "rule_score", "fraud_intensity_score",
                "sender_acct_txn_count_24h", "sender_acct_outflow_amt_24h",
                "sender_acct_unique_counterparties_7d", "ip_risk_score"]

existing_keys = [f for f in key_features if f in df.columns]
print(f"\n  {'Feature':<45s} │ {'AML Mean':>10s} {'Clean Mean':>10s} {'Ratio':>7s} │ {'AML Med':>10s} {'Clean Med':>10s}")
print("  " + "─" * 100)
for feat in existing_keys:
    am = df.loc[df["is_aml"]==1, feat].mean()
    cm = df.loc[df["is_aml"]==0, feat].mean()
    amed = df.loc[df["is_aml"]==1, feat].median()
    cmed = df.loc[df["is_aml"]==0, feat].median()
    ratio = am / max(cm, 0.0001)
    print(f"  {feat:<45s} │ {am:>10.2f} {cm:>10.2f} {ratio:>6.2f}x │ {amed:>10.2f} {cmed:>10.2f}")

# ── 3.6: Plots ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Initial EDA — Target & Key Features", fontsize=14, fontweight="bold")

# Target bar
ax = axes[0, 0]
colors = ["#2ecc71", "#e74c3c"]
target_counts.plot(kind="bar", ax=ax, color=colors)
ax.set_title("Target Distribution (is_aml)")
ax.set_xticklabels(["Clean (0)", "AML (1)"], rotation=0)
for i, v in enumerate(target_counts):
    ax.text(i, v + len(df)*0.01, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=9)

# FIS by AML status
ax = axes[0, 1]
if "fraud_intensity_score" in df.columns:
    df.loc[df["is_aml"]==0, "fraud_intensity_score"].hist(bins=50, alpha=0.6, ax=ax, label="Clean", color="#2ecc71", density=True)
    df.loc[df["is_aml"]==1, "fraud_intensity_score"].hist(bins=50, alpha=0.6, ax=ax, label="AML", color="#e74c3c", density=True)
    ax.set_title("FIS Distribution: Clean vs AML")
    ax.legend()

# Rule score by AML
ax = axes[1, 0]
if "rule_score" in df.columns:
    df.loc[df["is_aml"]==0, "rule_score"].hist(bins=50, alpha=0.6, ax=ax, label="Clean", color="#2ecc71", density=True)
    df.loc[df["is_aml"]==1, "rule_score"].hist(bins=50, alpha=0.6, ax=ax, label="AML", color="#e74c3c", density=True)
    ax.set_title("Rule Score Distribution: Clean vs AML")
    ax.legend()

# Alert level by AML
ax = axes[1, 1]
if "alert_level" in df.columns:
    ct = pd.crosstab(df["alert_level"], df["is_aml"], normalize="index") * 100
    ct = ct.reindex(["Critical", "High", "Medium", "Low", "None"])
    ct.plot(kind="barh", stacked=True, ax=ax, color=colors)
    ax.set_title("AML Rate by Alert Level")
    ax.set_xlabel("Percentage")
    ax.legend(["Clean", "AML"])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_initial_eda.png"), bbox_inches="tight")
plt.show()
print(f"\n  Saved: {OUTPUT_DIR}/01_initial_eda.png")



INITIAL EDA

── 3.1: Target Variable (is_aml) ──
  is_aml=0:    204,230 (61.2%)
  is_aml=1:    129,645 (38.8%)
  Imbalance ratio: 1.6:1 (Clean:AML)

── 3.2: Column Types ──
  Numeric: 226 | Object/String: 92 | Boolean: 0

── 3.3: Missing Values (top 20) ──
  No missing values found

── 3.4: Typology Distribution (within is_aml=1) ──
  Charity Abuse                              86,210 (66.5%)
  Funnel Account Network                     47,141 (36.4%)
  Third-Party Payment Web                    21,739 (16.8%)
  Pass-Through Transit Hub                    3,096 (2.4%)
  High-Risk Corridor Transfer                 3,047 (2.4%)
  Structuring (Smurfing)                      2,825 (2.2%)
  Rapid Multi-Hop Layering                    2,547 (2.0%)
  Underground Banking (Hawala)                1,704 (1.3%)
  Money Mule Network                          1,468 (1.1%)
  Circular Transaction Loop                   1,144 (0.9%)

── 3.5: Key Feature Statistics (AML vs Clean) ──

  Feature            

## 4 — Feature Classification
Features split into 3 tiers:
1. **Protected** (always included): Encoded categoricals + core scores
2. **Selectable** (feature selection applied): **Rule flags (126) + Graph/velocity/balance/IP**
3. **Excluded**: Labels, IDs, post-hoc scores, typology leakage

**KEY CHANGE**: Rule flags moved from Protected → Selectable. Feature importance will identify which rules actually matter.


In [4]:
print("=" * 90)
print("FEATURE CLASSIFICATION — Rules Now Under Selection")
print("=" * 90)

# ═══ TIER 3: EXCLUDED ═══
LABEL_COLS = {"is_aml", "is_aml_typology", "aml_typology", "typology_group_id", "aml_flag_source"}

ID_COLS = {
    "transaction_id", "timestamp", "datestamp", "customer_account_number",
    "customer_cif_id", "counterparty_account_number", "customer_name",
    "counterparty_name", "merchant_id", "merchant_name", "merchant_location",
    "session_id", "device_id_fingerprint", "ip_address", "pan", "aadhaar_number",
    "mobile_number", "email_id", "wallet_account_id", "beneficiary_wallet_id_vpa",
    "load_source_account_card_details", "customer_branch_ifsc_code",
    "counterparty_branch_ifsc_swift", "customer_cif_creation_date",
    "kyc_update_date", "account_wallet_opening_date", "account_wallet_inoperative_date",
    "date_of_birth", "date_of_incorporation",
    "father_spouse_name", "identification_proof_doc_no", "entity_identification_proof_doc_no",
    "cif_beneficial_owners", "name_beneficial_owners",
    "address_registered_office", "address_place_of_business",
    "address_beneficial_owners", "address_individual_customer",
    "place_of_incorporation", "browser_app_information",
    "geo_location_city_country", "escrow_account_linked",
    "gps_coordinates_lat", "gps_coordinates_lon",
    "customer_address_lat", "customer_address_lon"
}

POSTHOC_COLS = {
    "fraud_intensity_score", "fraud_intensity_score_raw", "fis_band",
    "alert_level", "rules_triggered", "rules_triggered_count",
    "predicted_aml", "predicted_typology", "typology_confidence"
}
POSTHOC_COLS.update({c for c in df.columns if c.startswith("prob_")})

TYPOLOGY_LEAKAGE_COLS = {c for c in df.columns if any(c.startswith(p) for p in
    ["typology_signal", "ts_", "convergence_risk", "cr_", "temporal_risk", "tr_"])}
TYPOLOGY_LEAKAGE_COLS.update({"typology_signal", "convergence_risk", "temporal_risk"})

INTERNAL_COLS = {c for c in df.columns if c.startswith("_")}

all_exclude = LABEL_COLS | ID_COLS | POSTHOC_COLS | TYPOLOGY_LEAKAGE_COLS | INTERNAL_COLS

# ═══ TIER 1: PROTECTED — Encoded categoricals + core scores only ═══
core_scores = [c for c in ["rule_score", "transaction_amount", "annual_income",
               "credit_summation_period", "debit_summation_period",
               "professional_experience_years"] if c in df.columns and c not in all_exclude]

STRING_FEATURES = sorted([c for c in [
    "transaction_type_dr_cr", "transaction_mode_channel_bank", "cash_flag",
    "transaction_type_ppi", "transaction_mode_channel_ppi", "transaction_status",
    "account_wallet_status", "pep_flag", "hni_flag", "minor_flag",
    "customer_type", "customer_entity_type", "account_category", "account_type",
    "customer_occupation_industry", "vkyc_flag", "wallet_kyc_category",
    "vpn_flag", "emulator_flag", "refund_chargeback_flag",
    "customer_current_risk_score", "tax_residency", "residency",
    "nationality", "citizenship", "non_face_to_face_flag",
    "merchant_category_code", "load_instrument_type", "authentication_method",
    "beneficial_owner_types", "passive_nfe", "source_of_funds",
    "source_of_funds_wallet", "currency"
] if c in df.columns and c not in all_exclude])

PROTECTED_NUMERIC = core_scores  # Rules REMOVED from protected

# ═══ TIER 2: SELECTABLE — Rules + Graph/Velocity/Balance/IP ═══
# Rules are now SELECTABLE — importance-based selection will keep only the useful ones
rule_flags = sorted([c for c in df.columns if c.startswith("rule_") and c not in
              {"rule_score", "rules_triggered", "rules_triggered_count"}
              and c not in all_exclude])

SELECTABLE_PREFIXES = [
    "sender_acct_", "sender_cust_", "sender_running", "sender_daily",
    "sender_balance", "sender_pct_", "sender_cumulative",
    "receiver_acct_", "receiver_running", "receiver_balance",
    "inflow_outflow_", "ip_risk", "ip_flag", "ip_txn", "ip_unique", "ip_cross"
]

graph_features = sorted([c for c in df.select_dtypes(include=[np.number]).columns
                          if c not in all_exclude and c not in PROTECTED_NUMERIC
                          and c not in rule_flags
                          and any(c.startswith(p) for p in SELECTABLE_PREFIXES)])

selectable_features = rule_flags + graph_features  # Rules + Graph combined

print(f"\n  ┌─────────────────────────────────────────────────────────────────────────┐")
print(f"  │  TIER 1 — PROTECTED (always included, no selection)                    │")
print(f"  │    Core numeric scores:          {len(core_scores):>4d}                                  │")
print(f"  │    Categorical (to encode):      {len(STRING_FEATURES):>4d}                                  │")
print(f"  ├─────────────────────────────────────────────────────────────────────────┤")
print(f"  │  TIER 2 — SELECTABLE (feature selection applied)                       │")
print(f"  │    Rule flags (126 binary):      {len(rule_flags):>4d}  ← NOW UNDER SELECTION            │")
print(f"  │    Graph/velocity/balance/IP:    {len(graph_features):>4d}                                  │")
print(f"  │    Total selectable:             {len(selectable_features):>4d}                                  │")
print(f"  ├─────────────────────────────────────────────────────────────────────────┤")
print(f"  │  TIER 3 — EXCLUDED                                                     │")
print(f"  │    Labels + IDs + Post-hoc:      {len(LABEL_COLS)+len(ID_COLS)+len(POSTHOC_COLS):>4d}                                  │")
print(f"  │    Typology leakage:             {len(TYPOLOGY_LEAKAGE_COLS):>4d}                                  │")
print(f"  └─────────────────────────────────────────────────────────────────────────┘")
print(f"\n  KEY CHANGE: 126 rule flags moved from Protected → Selectable")
print(f"  Feature selection will keep only rules that genuinely improve model recall")



FEATURE CLASSIFICATION — Rules Now Under Selection

  ┌─────────────────────────────────────────────────────────────────────────┐
  │  TIER 1 — PROTECTED (always included, no selection)                    │
  │    Core numeric scores:             6                                  │
  │    Categorical (to encode):        34                                  │
  ├─────────────────────────────────────────────────────────────────────────┤
  │  TIER 2 — SELECTABLE (feature selection applied)                       │
  │    Rule flags (126 binary):       126  ← NOW UNDER SELECTION            │
  │    Graph/velocity/balance/IP:      78                                  │
  │    Total selectable:              204                                  │
  ├─────────────────────────────────────────────────────────────────────────┤
  │  TIER 3 — EXCLUDED                                                     │
  │    Labels + IDs + Post-hoc:        60                                  │
  │    Typology leak

## 5 — Encode Categorical Features & Save Mappings


In [5]:
print("Encoding categorical features...")
import pickle

df_ml = df.copy()
encoded_cols = []
encoding_maps = {}

for col in STRING_FEATURES:
    if col not in df_ml.columns:
        continue
    vals = df_ml[col].astype(str).str.strip().str.upper()
    vals = vals.replace({"NAN": "", "NONE": "", "": "MISSING"})
    categories = sorted(vals.unique())
    cat_map = {cat: i for i, cat in enumerate(categories)}
    encoded_col = f"{col}_enc"
    df_ml[encoded_col] = vals.map(cat_map).fillna(-1).astype(int)
    encoded_cols.append(encoded_col)
    encoding_maps[col] = cat_map
    print(f"  {col:<45s} → {encoded_col:<50s} ({len(categories)} categories)")

print(f"\n  Total encoded columns: {len(encoded_cols)}")

# Save encoding maps
import json as _json
json_path = os.path.join(OUTPUT_DIR, "label_encoding_maps.json")
with open(json_path, "w") as f:
    _json.dump({col: {str(k): int(v) for k, v in m.items()} for col, m in encoding_maps.items()}, f, indent=2)
pkl_path = os.path.join(OUTPUT_DIR, "label_encoding_maps.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(encoding_maps, f)
enc_csv_dir = os.path.join(OUTPUT_DIR, "encoding_csvs")
os.makedirs(enc_csv_dir, exist_ok=True)
for col, mapping in encoding_maps.items():
    pd.DataFrame([{"category": k, "encoded_value": v} for k, v in mapping.items()]).sort_values("encoded_value").to_csv(
        os.path.join(enc_csv_dir, f"{col}_encoding.csv"), index=False)
print(f"  Saved: {json_path}, {pkl_path}, {enc_csv_dir}/ ({len(encoding_maps)} CSVs)")

# Build PROTECTED features (rule flags + core scores + encoded categoricals)
PROTECTED_FEATURES = PROTECTED_NUMERIC + encoded_cols
print(f"\n  PROTECTED features (always in model): {len(PROTECTED_FEATURES)}")
print(f"    Rule flags: {len(rule_flags)} | Core scores: {len(core_scores)} | Encoded cats: {len(encoded_cols)}")



Encoding categorical features...
  account_category                              → account_category_enc                               (7 categories)
  account_type                                  → account_type_enc                                   (5 categories)
  account_wallet_status                         → account_wallet_status_enc                          (2 categories)
  authentication_method                         → authentication_method_enc                          (5 categories)
  beneficial_owner_types                        → beneficial_owner_types_enc                         (4 categories)
  cash_flag                                     → cash_flag_enc                                      (2 categories)
  citizenship                                   → citizenship_enc                                    (15 categories)
  currency                                      → currency_enc                                       (6 categories)
  customer_current_risk_score         

## 6 — Correlation Analysis (Selectable Features Only)


In [6]:
print("=" * 90)
print("CORRELATION ANALYSIS — SELECTABLE Features vs is_aml")
print("(Rule flags + categoricals are PROTECTED and skip this step)")
print("=" * 90)

target = df_ml["is_aml"].astype(float)

# Correlations for SELECTABLE features only
sel_correlations = {}
for feat in selectable_features:
    if feat not in df_ml.columns: continue
    vals = pd.to_numeric(df_ml[feat], errors="coerce").fillna(0)
    if vals.std() == 0:
        sel_correlations[feat] = 0.0
        continue
    sel_correlations[feat] = vals.corr(target)

corr_df = pd.DataFrame([
    {"feature": k, "correlation": v, "abs_correlation": abs(v)}
    for k, v in sel_correlations.items()
]).sort_values("abs_correlation", ascending=False)

# Also compute correlations for PROTECTED features (for reporting only, no selection)
prot_correlations = {}
for feat in PROTECTED_FEATURES:
    if feat not in df_ml.columns: continue
    vals = pd.to_numeric(df_ml[feat], errors="coerce").fillna(0)
    if vals.std() == 0: prot_correlations[feat] = 0.0; continue
    prot_correlations[feat] = vals.corr(target)

prot_corr_df = pd.DataFrame([
    {"feature": k, "correlation": v, "abs_correlation": abs(v)}
    for k, v in prot_correlations.items()
]).sort_values("abs_correlation", ascending=False)

print(f"\n── 6.1: Top 30 SELECTABLE Features by Correlation with is_aml ──")
print(f"  (These are the features subject to selection)\n")
print(f"  {'Rank':<5s} {'Feature':<55s} {'Correlation':>12s} {'Signal':>8s}")
print("  " + "─" * 83)
for i, (_, row) in enumerate(corr_df.head(30).iterrows(), 1):
    strength = "STRONG" if row["abs_correlation"] > 0.1 else ("MEDIUM" if row["abs_correlation"] > 0.05 else "WEAK")
    bar = "█" * int(row["abs_correlation"] * 200)
    print(f"  {i:<5d} {row['feature']:<55s} {row['correlation']:>+11.6f} {strength:<8s} {bar}")

print(f"\n── 6.2: Top 20 PROTECTED Features by Correlation (for reference, NOT selected out) ──\n")
print(f"  {'Rank':<5s} {'Feature':<55s} {'Correlation':>12s} {'Status':>10s}")
print("  " + "─" * 85)
for i, (_, row) in enumerate(prot_corr_df.head(20).iterrows(), 1):
    print(f"  {i:<5d} {row['feature']:<55s} {row['correlation']:>+11.6f} {'PROTECTED':>10s}")

# Heatmap (top selectable + top protected)
top_sel = corr_df.head(15)["feature"].tolist()
top_prot = prot_corr_df.head(5)["feature"].tolist()
heatmap_feats = top_sel + top_prot + ["is_aml"]
heatmap_feats = [c for c in heatmap_feats if c in df_ml.columns]

fig, ax = plt.subplots(figsize=(16, 14))
hm = df_ml[heatmap_feats].corr()
mask = np.triu(np.ones_like(hm, dtype=bool))
sns.heatmap(hm, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            ax=ax, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Top Selectable + Protected Features — Correlation Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_correlation_heatmap.png"), bbox_inches="tight")
plt.show()
print(f"\n  Saved: {OUTPUT_DIR}/02_correlation_heatmap.png")
corr_df.to_csv(os.path.join(OUTPUT_DIR, "selectable_feature_correlations.csv"), index=False)



CORRELATION ANALYSIS — SELECTABLE Features vs is_aml
(Rule flags + categoricals are PROTECTED and skip this step)

── 6.1: Top 30 SELECTABLE Features by Correlation with is_aml ──
  (These are the features subject to selection)

  Rank  Feature                                                  Correlation   Signal
  ───────────────────────────────────────────────────────────────────────────────────
  1     sender_acct_txn_count_30d                                 -0.262509 STRONG   ████████████████████████████████████████████████████
  2     sender_acct_outflow_count_30d                             -0.258154 STRONG   ███████████████████████████████████████████████████
  3     sender_cust_outflow_count_30d                             -0.245337 STRONG   █████████████████████████████████████████████████
  4     sender_cust_txn_count_30d                                 -0.244617 STRONG   ████████████████████████████████████████████████
  5     sender_acct_inflow_count_30d                   

## 7 — Multicollinearity & VIF (Selectable Features Only)


In [7]:
print("=" * 90)
print("MULTICOLLINEARITY & VIF — Selectable Features Only")
print("(Rule flags + categoricals are PROTECTED, not checked here)")
print("=" * 90)

from sklearn.linear_model import LinearRegression

THRESHOLD = 0.90
valid_sel = [f for f in selectable_features if f in df_ml.columns
             and pd.to_numeric(df_ml[f], errors="coerce").std() > 0]

print(f"  Computing pairwise correlations for {len(valid_sel)} selectable features...")
if len(df_ml) > 50000:
    sample_df = df_ml[valid_sel].sample(50000, random_state=42)
else:
    sample_df = df_ml[valid_sel].copy()
for c in sample_df.columns:
    sample_df[c] = pd.to_numeric(sample_df[c], errors="coerce").fillna(0)

corr_all = sample_df.corr()

# Find pairs
high_corr_pairs = []
for i in range(len(corr_all.columns)):
    for j in range(i+1, len(corr_all.columns)):
        r = corr_all.iloc[i, j]
        if abs(r) >= THRESHOLD:
            high_corr_pairs.append((corr_all.columns[i], corr_all.columns[j], r))
high_corr_pairs.sort(key=lambda x: -abs(x[2]))

target_corr = {row["feature"]: row["abs_correlation"] for _, row in corr_df.iterrows()}

print(f"\n── 7.1: Highly Correlated Selectable Pairs (|r| >= {THRESHOLD}) ──")
print(f"  Found: {len(high_corr_pairs)} pairs\n")
print(f"  {'Feature A':<45s} {'Feature B':<45s} {'Corr':>8s} {'TgtCorr A':>10s} {'TgtCorr B':>10s} {'Recommendation':>20s}")
print("  " + "─" * 142)
for f1, f2, r in high_corr_pairs[:40]:
    c1 = target_corr.get(f1, 0); c2 = target_corr.get(f2, 0)
    rec = f"Drop {f2[:18]}" if c1 >= c2 else f"Drop {f1[:18]}"
    print(f"  {f1:<45s} {f2:<45s} {r:>+7.4f} {c1:>9.6f} {c2:>9.6f}   {rec}")

# VIF
print(f"\n── 7.2: VIF Scores (Selectable Features) ──")
top_vif = corr_df.head(min(50, len(valid_sel)))["feature"].tolist()
top_vif = [f for f in top_vif if f in sample_df.columns]
print(f"  Computing VIF for {len(top_vif)} features...")

X_vif = sample_df[top_vif].copy()
X_vif = (X_vif - X_vif.mean()) / X_vif.std().replace(0, 1)

vif_results = []
lr = LinearRegression()
for i, feat in enumerate(top_vif):
    y_vif = X_vif[feat].values; X_others = X_vif.drop(columns=[feat]).values
    try:
        lr.fit(X_others, y_vif); r2 = lr.score(X_others, y_vif)
        vif = 1 / max(1 - r2, 0.0001)
    except: vif = float("inf"); r2 = 0
    vif_results.append({"feature": feat, "vif": vif, "r_squared": r2, "target_corr": target_corr.get(feat, 0)})

vif_df = pd.DataFrame(vif_results).sort_values("vif", ascending=False)

print(f"\n  {'Rank':<5s} {'Feature':<50s} {'VIF':>10s} {'R²':>8s} {'Target Corr':>12s} {'Severity':>12s}")
print("  " + "─" * 102)
for i, (_, row) in enumerate(vif_df.iterrows(), 1):
    v = row["vif"]
    sev = "⚠ CRITICAL" if v>=50 else ("⚡ SEVERE" if v>=10 else ("● MODERATE" if v>=5 else "✓ OK"))
    vd = f"{v:>10.2f}" if v < 10000 else f"{v:>10.0f}"
    print(f"  {i:<5d} {row['feature']:<50s} {vd} {row['r_squared']:>7.4f} {row['target_corr']:>11.6f} {sev}")

# Summary
crit=len(vif_df[vif_df["vif"]>=50]); sev=len(vif_df[(vif_df["vif"]>=10)&(vif_df["vif"]<50)])
mod=len(vif_df[(vif_df["vif"]>=5)&(vif_df["vif"]<10)]); ok=len(vif_df[vif_df["vif"]<5])
print(f"\n  VIF Summary: ✓ OK={ok} | ● Moderate={mod} | ⚡ Severe={sev} | ⚠ Critical={crit}")

vif_df.to_csv(os.path.join(OUTPUT_DIR, "vif_scores_selectable.csv"), index=False)

# Greedy removal (on selectable only)
to_remove = set()
for f1, f2, r in high_corr_pairs:
    if f1 in to_remove or f2 in to_remove: continue
    c1 = target_corr.get(f1, 0); c2 = target_corr.get(f2, 0)
    to_remove.add(f2 if c1 >= c2 else f1)

selectable_after_multicollinearity = [f for f in selectable_features if f not in to_remove]
print(f"\n  Selectable before: {len(selectable_features)} → after multicollinearity: {len(selectable_after_multicollinearity)} (removed {len(to_remove)})")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
top30_vif = vif_df.head(30).sort_values("vif")
colors = ["#e74c3c" if v>=50 else "#f39c12" if v>=10 else "#3498db" if v>=5 else "#2ecc71" for v in top30_vif["vif"]]
axes[0].barh(top30_vif["feature"], top30_vif["vif"], color=colors)
axes[0].axvline(x=5, color="orange", linestyle="--", alpha=0.7); axes[0].axvline(x=10, color="red", linestyle="--", alpha=0.7)
axes[0].set_title("VIF Scores (Selectable Features)", fontsize=12, fontweight="bold")
sizes = [ok, mod, sev, crit]; labels = [f"OK<5 ({ok})", f"Mod 5-10 ({mod})", f"Sev 10-50 ({sev})", f"Crit 50+ ({crit})"]
axes[1].pie([s for s in sizes if s>0], labels=[l for l,s in zip(labels,sizes) if s>0],
            colors=["#2ecc71","#3498db","#f39c12","#e74c3c"][:sum(1 for s in sizes if s>0)], autopct="%1.0f%%")
axes[1].set_title("VIF Severity Distribution", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_vif_analysis.png"), bbox_inches="tight")
plt.show()



MULTICOLLINEARITY & VIF — Selectable Features Only
(Rule flags + categoricals are PROTECTED, not checked here)
  Computing pairwise correlations for 130 selectable features...

── 7.1: Highly Correlated Selectable Pairs (|r| >= 0.9) ──
  Found: 13 pairs

  Feature A                                     Feature B                                         Corr  TgtCorr A  TgtCorr B       Recommendation
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  sender_balance_after_txn                      sender_balance_before_txn                     +0.9871  0.106140  0.112076   Drop sender_balance_aft
  receiver_balance_after_txn                    receiver_balance_before_txn                   +0.9868  0.109956  0.111851   Drop receiver_balance_a
  rule_ppi_kyc_behavior_breach                  rule_ppi_kyc_expiry_limit                     +0.9836  0.008380  0.009060   Drop rule_ppi_kyc_behav
  sender_c

## 8 — Feature Importance (Rules + Graph Features Combined)
LightGBM importance scan on ALL selectable features (rules + graph).
Rules with zero importance are removed — they don't help the model distinguish AML from clean.


In [9]:
print("=" * 90)
print("FEATURE IMPORTANCE — Rules + Graph Features Combined")
print("=" * 90)
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("  LightGBM not installed. Run: pip install lightgbm")
    
if HAS_LGB:
    # Run importance on ALL selectable features (rules + graph combined)
    X_sel = df_ml[selectable_after_multicollinearity].copy()
    for c in X_sel.columns: X_sel[c] = pd.to_numeric(X_sel[c], errors="coerce").fillna(0)
    y_sel = df_ml["is_aml"].astype(int)

    train_ds = lgb.Dataset(X_sel, label=y_sel)
    params = {"objective":"binary","metric":"auc","learning_rate":0.1,"num_leaves":31,
              "max_depth":6,"min_child_samples":50,"subsample":0.8,"colsample_bytree":0.8,
              "scale_pos_weight":(y_sel==0).sum()/max((y_sel==1).sum(),1),
              "verbosity":-1,"random_state":42,"n_jobs":-1}

    print("  Training LightGBM for feature importance (rules + graph)...")
    model_imp = lgb.train(params, train_ds, num_boost_round=300)

    importance_gain = pd.DataFrame({
        "feature": selectable_after_multicollinearity,
        "gain": model_imp.feature_importance(importance_type="gain"),
        "split": model_imp.feature_importance(importance_type="split"),
        "is_rule": [f.startswith("rule_") for f in selectable_after_multicollinearity]
    }).sort_values("gain", ascending=False)

    # Separate analysis for rules vs graph
    rule_importance = importance_gain[importance_gain["is_rule"]].copy()
    graph_importance = importance_gain[~importance_gain["is_rule"]].copy()

    print(f"\n── Rule Flag Importance (Top 30 of {len(rule_importance)}) ──")
    print(f"  {'Rank':<5s} {'Rule':<55s} {'Gain':>12s} {'Splits':>8s}")
    print("  " + "─" * 83)
    for i, (_, row) in enumerate(rule_importance.head(30).iterrows(), 1):
        print(f"  {i:<5d} {row['feature']:<55s} {row['gain']:>12.1f} {row['split']:>8.0f}")

    zero_rules = rule_importance[rule_importance["gain"] == 0]
    active_rules = rule_importance[rule_importance["gain"] > 0]
    print(f"\n  Rules with importance > 0: {len(active_rules)} (KEPT)")
    print(f"  Rules with zero importance: {len(zero_rules)} (REMOVED)")
    if len(zero_rules) > 0:
        print(f"  Removed rules: {', '.join(zero_rules['feature'].tolist()[:10])}{'...' if len(zero_rules) > 10 else ''}")

    print(f"\n── Graph/Velocity Feature Importance (Top 20 of {len(graph_importance)}) ──")
    print(f"  {'Rank':<5s} {'Feature':<55s} {'Gain':>12s} {'Splits':>8s}")
    print("  " + "─" * 83)
    for i, (_, row) in enumerate(graph_importance.head(20).iterrows(), 1):
        print(f"  {i:<5d} {row['feature']:<55s} {row['gain']:>12.1f} {row['split']:>8.0f}")

    zero_graph = graph_importance[graph_importance["gain"] == 0]
    active_graph = graph_importance[graph_importance["gain"] > 0]

    # Combined selection
    selected_features_all = importance_gain[importance_gain["gain"] > 0]["feature"].tolist()
    selected_rules = active_rules["feature"].tolist()
    selected_graph_features = active_graph["feature"].tolist()

    print(f"\n  ── Selection Summary ──")
    print(f"  Rules:  {len(rule_flags)} started → {len(selected_rules)} selected ({len(zero_rules)} removed)")
    print(f"  Graph:  {len(graph_features)} started → {len(selected_graph_features)} selected ({len(zero_graph)} removed)")
    print(f"  Total selected: {len(selected_features_all)}")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(18, 10))
    top30_rules = rule_importance.head(30).sort_values("gain")
    axes[0].barh(top30_rules["feature"], top30_rules["gain"], color="#e74c3c")
    axes[0].set_title(f"Top 30 Rules by Importance ({len(selected_rules)} kept / {len(zero_rules)} removed)", fontsize=11, fontweight="bold")

    top20_graph = graph_importance.head(20).sort_values("gain")
    axes[1].barh(top20_graph["feature"], top20_graph["gain"], color="#2ecc71")
    axes[1].set_title(f"Top 20 Graph Features ({len(selected_graph_features)} kept / {len(zero_graph)} removed)", fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "03_feature_importance_rules_graph.png"), bbox_inches="tight")
    plt.show()

    importance_gain.to_csv(os.path.join(OUTPUT_DIR, "feature_importance_all.csv"), index=False)
    print(f"  Saved: feature_importance_all.csv + 03_feature_importance_rules_graph.png")
else:
    selected_features_all = selectable_after_multicollinearity
    selected_rules = rule_flags
    selected_graph_features = graph_features



FEATURE IMPORTANCE — Rules + Graph Features Combined
  Training LightGBM for feature importance (rules + graph)...

── Rule Flag Importance (Top 30 of 123) ──
  Rank  Rule                                                            Gain   Splits
  ───────────────────────────────────────────────────────────────────────────────────
  1     rule_attempted_failed                                       136138.6       69
  2     rule_structuring_pattern                                      3612.2       75
  3     rule_negative_list_country                                    2347.4        9
  4     rule_unusual_type_spike                                       1095.8       16
  5     rule_bo_address_mismatch                                       937.9       19
  6     rule_large_cash_individual                                     752.7       21
  7     rule_low_income_large                                          740.3       24
  8     rule_large_intra_individual                                

## 9 — Feature Selection Summary & Final Feature Set


In [24]:
print("=" * 90)
print("FEATURE SELECTION SUMMARY")
print("=" * 90)

# Final = PROTECTED (categoricals + core) + SELECTED (surviving rules + graph)
features_final = PROTECTED_FEATURES + selected_features_all
features_final = list(dict.fromkeys(features_final))

print(f"""
  ┌──────────────────────────────────────────────────────────────────────────────┐
  │  FINAL FEATURE SET                                                          │
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  PROTECTED (always included):                                               │
  │    Encoded categoricals:         {len(encoded_cols):>4d}                                       │
  │    Core numeric scores:          {len(core_scores):>4d}                                       │
  │    Subtotal PROTECTED:           {len(PROTECTED_FEATURES):>4d}                                       │
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  SELECTED (survived importance-based selection):                            │
  │    Rule flags:                   {len(selected_rules):>4d}  (from 126 → only important ones)   │
  │    Graph/velocity/balance/IP:    {len(selected_graph_features):>4d}                                       │
  │    Subtotal SELECTED:            {len(selected_features_all):>4d}                                       │
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  TOTAL FEATURES IN MODEL:       {len(features_final):>4d}                                       │
  └──────────────────────────────────────────────────────────────────────────────┘
""")

# Show which rules were kept vs removed
print(f"  Rules KEPT ({len(selected_rules)}):")
for r in selected_rules[:20]:
    print(f"    ✓ {r}")
if len(selected_rules) > 20:
    print(f"    ... and {len(selected_rules) - 20} more")

removed_rules = [r for r in rule_flags if r not in selected_rules]
print(f"\n  Rules REMOVED ({len(removed_rules)} — zero importance to model):")
for r in removed_rules[:15]:
    print(f"    ✗ {r}")
if len(removed_rules) > 15:
    print(f"    ... and {len(removed_rules) - 15} more")

with open(os.path.join(OUTPUT_DIR, "selected_features.txt"), "w") as f:
    for feat in features_final: f.write(feat + "\n")
with open(os.path.join(OUTPUT_DIR, "selected_rules.txt"), "w") as f:
    for feat in selected_rules: f.write(feat + "\n")
with open(os.path.join(OUTPUT_DIR, "removed_rules.txt"), "w") as f:
    for feat in removed_rules: f.write(feat + "\n")
print(f"\n  Saved: selected_features.txt, selected_rules.txt, removed_rules.txt")



FEATURE SELECTION SUMMARY

  ┌──────────────────────────────────────────────────────────────────────────────┐
  │  FINAL FEATURE SET                                                          │
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  PROTECTED (always included):                                               │
  │    Encoded categoricals:           34                                       │
  │    Core numeric scores:             6                                       │
  │    Subtotal PROTECTED:             40                                       │
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  SELECTED (survived importance-based selection):                            │
  │    Rule flags:                     31  (from 126 → only important ones)   │
  │    Graph/velocity/balance/IP:      65                                       │
  │    Subtotal SELECTED:              96                             

In [29]:
print("=" * 90)
print("CLASS IMBALANCE ANALYSIS & HANDLING")
print("=" * 90)

from sklearn.model_selection import train_test_split

# Prepare data
X = df_ml[features_final].copy()
for c in X.columns: X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)
y = df_ml["is_aml"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()
imbalance_ratio = n_neg / max(n_pos, 1)

print(f"\n  Training set class distribution:")
print(f"    Clean (0): {n_neg:>10,} ({n_neg/len(y_train)*100:.1f}%)")
print(f"    AML   (1): {n_pos:>10,} ({n_pos/len(y_train)*100:.1f}%)")
print(f"    Imbalance ratio: {imbalance_ratio:.2f}:1")

# ── Strategy 1: Class Weights ──
print(f"\n── Strategy 1: Class Weights (scale_pos_weight = {imbalance_ratio:.2f}) ──")
print(f"  LightGBM multiplies the loss for AML class by {imbalance_ratio:.2f}x")
print(f"  Effect: Model penalized {imbalance_ratio:.1f}x more for missing an AML transaction")

# ── Strategy 2: SMOTE ──
print(f"\n── Strategy 2: SMOTE (Synthetic Minority Oversampling) ──")
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    print("  imblearn not installed. Run: pip install imbalanced-learn")

if HAS_SMOTE:
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    print(f"  Before SMOTE: {len(X_train):,} rows (AML={n_pos:,})")
    print(f"  After SMOTE:  {len(X_train_smote):,} rows (AML={(y_train_smote==1).sum():,})")
    print(f"  Synthetic AML samples added: {(y_train_smote==1).sum() - n_pos:,}")
else:
    X_train_smote = X_train; y_train_smote = y_train

# ── Strategy 3: Threshold Tuning ──
print(f"\n── Strategy 3: Threshold Tuning ──")
print(f"  Default threshold: 0.5 (equal weight to precision and recall)")
print(f"  For AML: lower threshold (e.g., 0.3) increases recall at cost of precision")
print(f"  Will test thresholds: 0.3, 0.4, 0.5, 0.6, 0.7 after model training")

# ── Compare all 3 strategies ──
if HAS_LGB:
    from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

    results = {}
    
    configs = {
        "Baseline (no handling)": {
            "X": X_train, "y": y_train,
            "params": {"scale_pos_weight": 1.0}
        },
        "Class Weights": {
            "X": X_train, "y": y_train,
            "params": {"scale_pos_weight": imbalance_ratio}
        },
        "SMOTE": {
            "X": X_train_smote, "y": y_train_smote,
            "params": {"scale_pos_weight": 1.0}
        },
        "Weights + SMOTE": {
            "X": X_train_smote, "y": y_train_smote,
            "params": {"scale_pos_weight": imbalance_ratio * 0.5}  # reduced since SMOTE already balances
        },
    }
    
    print(f"\n── Comparing Imbalance Strategies ──\n")
    print(f"  {'Strategy':<25s} │ {'AUC':>7s} {'F1':>7s} {'Prec':>7s} {'Recall':>7s} │ {'TP':>8s} {'FP':>8s} {'FN':>8s} {'TN':>8s}")
    print("  " + "─" * 95)
    
    for name, cfg in configs.items():
        base_params = {
            "objective":"binary","metric":"auc","learning_rate":0.05,
            "num_leaves":63,"max_depth":8,"min_child_samples":50,
            "subsample":0.8,"colsample_bytree":0.8,
            "verbosity":-1,"random_state":42,"n_jobs":-1
        }
        base_params.update(cfg["params"])
        
        ds = lgb.Dataset(cfg["X"], label=cfg["y"])
        val_ds = lgb.Dataset(X_test, label=y_test, reference=ds)
        
        model = lgb.train(base_params, ds, num_boost_round=300,
                          valid_sets=[val_ds], callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)])
        
        y_prob = model.predict(X_test)
        y_pred = (y_prob >= 0.5).astype(int)
        
        auc = roc_auc_score(y_test, y_prob)
        f1 = f1_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y_test, y_pred)
        tp = cm[1,1]; fp = cm[0,1]; fn = cm[1,0]; tn = cm[0,0]
        
        results[name] = {"auc": auc, "f1": f1, "precision": prec, "recall": rec,
                         "model": model, "y_prob": y_prob, "tp": tp, "fp": fp, "fn": fn, "tn": tn}
        
        print(f"  {name:<25s} │ {auc:>6.4f} {f1:>6.4f} {prec:>6.4f} {rec:>6.4f} │ {tp:>8,} {fp:>8,} {fn:>8,} {tn:>8,}")
    
    # Find best strategy
    best_strategy = max(results, key=lambda k: results[k]["f1"])
    print(f"\n  ► Best strategy by F1: {best_strategy} (F1={results[best_strategy]['f1']:.4f})")
    
    # ── Threshold tuning on best model ──
    best_prob = results[best_strategy]["y_prob"]
    
    print(f"\n── Threshold Tuning on {best_strategy} ──\n")
    print(f"  {'Threshold':>10s} │ {'F1':>7s} {'Prec':>7s} {'Recall':>7s} │ {'TP':>8s} {'FP':>8s} {'FN':>8s}")
    print("  " + "─" * 65)
    
    best_f1 = 0; best_thresh = 0.5
    for thresh in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]:
        yp = (best_prob >= thresh).astype(int)
        f1 = f1_score(y_test, yp); p = precision_score(y_test, yp, zero_division=0); r = recall_score(y_test, yp)
        cm = confusion_matrix(y_test, yp)
        marker = " ◄" if f1 > best_f1 else ""
        if f1 > best_f1: best_f1 = f1; best_thresh = thresh
        print(f"  {thresh:>10.2f} │ {f1:>6.4f} {p:>6.4f} {r:>6.4f} │ {cm[1,1]:>8,} {cm[0,1]:>8,} {cm[1,0]:>8,}{marker}")
    
    print(f"\n  ► Optimal threshold: {best_thresh} (F1={best_f1:.4f})")
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Strategy comparison
    strats = list(results.keys())
    metrics = ["auc", "f1", "precision", "recall"]
    x = np.arange(len(strats)); width = 0.2
    for i, m in enumerate(metrics):
        vals = [results[s][m] for s in strats]
        axes[0].bar(x + i*width, vals, width, label=m.upper())
    axes[0].set_xticks(x + width*1.5); axes[0].set_xticklabels(strats, rotation=20, ha="right")
    axes[0].set_title("Imbalance Strategy Comparison", fontweight="bold"); axes[0].legend(); axes[0].set_ylim(0, 1.1)
    
    # Threshold curve
    thresholds = np.arange(0.1, 0.9, 0.02)
    f1s = [f1_score(y_test, (best_prob>=t).astype(int)) for t in thresholds]
    precs = [precision_score(y_test, (best_prob>=t).astype(int), zero_division=0) for t in thresholds]
    recs = [recall_score(y_test, (best_prob>=t).astype(int)) for t in thresholds]
    axes[1].plot(thresholds, f1s, "b-", lw=2, label="F1"); axes[1].plot(thresholds, precs, "g--", label="Precision")
    axes[1].plot(thresholds, recs, "r--", label="Recall"); axes[1].axvline(best_thresh, color="k", linestyle=":", alpha=0.5)
    axes[1].set_title("Threshold Tuning Curve", fontweight="bold"); axes[1].set_xlabel("Threshold"); axes[1].legend()
    
    # Score distribution
    axes[2].hist(best_prob[y_test==0], bins=50, alpha=0.6, label="Clean", color="#2ecc71", density=True)
    axes[2].hist(best_prob[y_test==1], bins=50, alpha=0.6, label="AML", color="#e74c3c", density=True)
    axes[2].axvline(best_thresh, color="k", linestyle="--", label=f"Threshold={best_thresh}")
    axes[2].set_title("Score Distribution", fontweight="bold"); axes[2].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "06_imbalance_handling.png"), bbox_inches="tight")
    plt.show()
    print(f"\n  Saved: {OUTPUT_DIR}/06_imbalance_handling.png")



CLASS IMBALANCE ANALYSIS & HANDLING

  Training set class distribution:
    Clean (0):    163,384 (61.2%)
    AML   (1):    103,716 (38.8%)
    Imbalance ratio: 1.58:1

── Strategy 1: Class Weights (scale_pos_weight = 1.58) ──
  LightGBM multiplies the loss for AML class by 1.58x
  Effect: Model penalized 1.6x more for missing an AML transaction

── Strategy 2: SMOTE (Synthetic Minority Oversampling) ──
  imblearn not installed. Run: pip install imbalanced-learn

── Strategy 3: Threshold Tuning ──
  Default threshold: 0.5 (equal weight to precision and recall)
  For AML: lower threshold (e.g., 0.3) increases recall at cost of precision
  Will test thresholds: 0.3, 0.4, 0.5, 0.6, 0.7 after model training

── Comparing Imbalance Strategies ──

  Strategy                  │     AUC      F1    Prec  Recall │       TP       FP       FN       TN
  ───────────────────────────────────────────────────────────────────────────────────────────────
Training until validation scores don't improve for

## 10 — Phase 1: Typology-Aware Split + Hyperparameter Tuning
- Stratified split preserving ALL typologies in both train and test
- 8 hyperparameter configurations tested
- Threshold tuning biased toward recall ≥ 0.85
- Selection criterion: 0.3×AUC + 0.3×F1 + 0.4×Recall (recall-heavy)


In [26]:
print("=" * 90)
print("CLASS IMBALANCE + HYPERPARAMETER TUNING + TYPOLOGY-AWARE SPLIT")
print("=" * 90)

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

# ═══ Prepare data ═══
X = df_ml[features_final].copy()
for c in X.columns: X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)
y = df_ml["is_aml"].astype(int)

# ═══ Typology-aware stratification ═══
print("\n── Typology-Aware Stratified Split ──")
typ_col = "aml_typology"
df_ml["_primary_typology"] = df_ml[typ_col].astype(str).apply(
    lambda x: x.split("; ")[0].strip() if x and x not in ("nan", "", "None") else "Clean"
)
df_ml["_strat_key"] = df_ml["_primary_typology"]
typ_counts = df_ml["_strat_key"].value_counts()
rare = typ_counts[typ_counts < 50].index.tolist()
if rare:
    df_ml.loc[df_ml["_strat_key"].isin(rare), "_strat_key"] = "Rare_Typology"
    print(f"  Merged {len(rare)} rare typologies for stratification: {rare}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=df_ml["_strat_key"]
)
train_indices = X_train.index; test_indices = X_test.index

n_pos = (y_train == 1).sum(); n_neg = (y_train == 0).sum()
imbalance_ratio = n_neg / max(n_pos, 1)
print(f"  Train: {len(X_train):,} | Test: {len(X_test):,} | Imbalance: {imbalance_ratio:.2f}:1")

# Verify ALL typologies in both sets
print(f"\n  {'Typology':<40s} {'Train':>8s} {'Test':>8s} {'Status':>8s}")
print(f"  {'─'*68}")
train_typs = df_ml.loc[train_indices, "_primary_typology"]
test_typs = df_ml.loc[test_indices, "_primary_typology"]
for typ in sorted(set(train_typs.unique()) | set(test_typs.unique())):
    tr = (train_typs == typ).sum(); te = (test_typs == typ).sum()
    status = "✓" if tr > 0 and te > 0 else "⚠ MISSING"
    print(f"  {typ:<40s} {tr:>8,} {te:>8,} {status}")

# ═══ SMOTE ═══
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    print(f"\n  SMOTE: {len(X_train):,} → {len(X_train_smote):,} (added {(y_train_smote==1).sum()-n_pos:,} synthetic AML)")
except ImportError:
    HAS_SMOTE = False; X_train_smote = X_train; y_train_smote = y_train

# ═══ Hyperparameter Tuning (8 configs) ═══
print(f"\n── Hyperparameter Tuning (8 configurations) ──\n")

tuning_configs = [
    {"name":"Baseline",          "nl":63,  "md":8,  "mc":50,  "lr":0.05, "ra":0,   "rl":0,   "sub":0.8,"col":0.8,"data":"orig","spw":imbalance_ratio},
    {"name":"Deep+Reg",          "nl":127, "md":10, "mc":30,  "lr":0.03, "ra":0.1, "rl":1.0, "sub":0.8,"col":0.8,"data":"orig","spw":imbalance_ratio},
    {"name":"Slow+Deep+L2",      "nl":127, "md":10, "mc":20,  "lr":0.01, "ra":0.1, "rl":2.0, "sub":0.7,"col":0.7,"data":"orig","spw":imbalance_ratio},
    {"name":"Wide+Shallow",      "nl":255, "md":6,  "mc":100, "lr":0.05, "ra":0.1, "rl":1.0, "sub":0.7,"col":0.8,"data":"orig","spw":imbalance_ratio},
    {"name":"SMOTE+Balanced",    "nl":63,  "md":8,  "mc":50,  "lr":0.05, "ra":0,   "rl":0,   "sub":0.8,"col":0.8,"data":"smote","spw":1.0},
    {"name":"SMOTE+Deep+Reg",    "nl":127, "md":10, "mc":30,  "lr":0.03, "ra":0.1, "rl":1.0, "sub":0.8,"col":0.8,"data":"smote","spw":1.0},
    {"name":"SMOTE+HalfWt",      "nl":127, "md":10, "mc":30,  "lr":0.03, "ra":0.1, "rl":1.0, "sub":0.8,"col":0.8,"data":"smote","spw":imbalance_ratio*0.5},
    {"name":"HeavyReg+Slow",     "nl":31,  "md":6,  "mc":100, "lr":0.01, "ra":0.5, "rl":5.0, "sub":0.6,"col":0.7,"data":"orig","spw":imbalance_ratio},
]

print(f"  {'Config':<22s} │ {'AUC':>7s} {'F1':>7s} {'Prec':>7s} {'Recall':>7s} │ {'Rounds':>6s} {'Thresh':>6s} {'F1@Thr':>7s}")
print("  " + "─" * 85)

results = {}

for cfg in tuning_configs:
    params = {"objective":"binary","metric":"auc","num_leaves":cfg["nl"],"max_depth":cfg["md"],
              "min_child_samples":cfg["mc"],"learning_rate":cfg["lr"],"reg_alpha":cfg["ra"],
              "reg_lambda":cfg["rl"],"subsample":cfg["sub"],"colsample_bytree":cfg["col"],
              "scale_pos_weight":cfg["spw"],"verbosity":-1,"random_state":42,"n_jobs":-1}
    
    Xtr = X_train_smote if cfg["data"]=="smote" else X_train
    ytr = y_train_smote if cfg["data"]=="smote" else y_train
    
    ds = lgb.Dataset(Xtr, label=ytr)
    val = lgb.Dataset(X_test, label=y_test, reference=ds)
    mdl = lgb.train(params, ds, num_boost_round=1000,
                    valid_sets=[val], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    
    prob = mdl.predict(X_test); auc_v = roc_auc_score(y_test, prob)
    bt=0.5; bf1=0
    for t in np.arange(0.25,0.75,0.02):
        f = f1_score(y_test,(prob>=t).astype(int))
        if f > bf1: bf1=f; bt=round(t,2)
    
    pred = (prob>=bt).astype(int)
    f1_v=f1_score(y_test,pred); p_v=precision_score(y_test,pred); r_v=recall_score(y_test,pred)
    cm=confusion_matrix(y_test,pred)
    
    results[cfg["name"]] = {"auc":auc_v,"f1":f1_v,"precision":p_v,"recall":r_v,
                            "model":mdl,"y_prob":prob,"threshold":bt,
                            "tp":cm[1,1],"fp":cm[0,1],"fn":cm[1,0],"tn":cm[0,0],
                            "config":cfg,"rounds":mdl.best_iteration if hasattr(mdl,"best_iteration") else 0}
    
    print(f"  {cfg['name']:<22s} │ {auc_v:>6.4f} {f1_v:>6.4f} {p_v:>6.4f} {r_v:>6.4f} │ {results[cfg['name']]['rounds']:>6} {bt:>5.2f} {bf1:>6.4f}")

# ═══ Config Selection: Best recall among configs with F1 within 95% of peak ═══
peak_f1_configs = max(v["f1"] for v in results.values())
f1_floor_configs = peak_f1_configs * 0.95

eligible_configs = {k: v for k, v in results.items() if v["f1"] >= f1_floor_configs}
best_strategy = max(eligible_configs, key=lambda k: eligible_configs[k]["recall"])
best_thresh = results[best_strategy]["threshold"]

print(f"\n  Config Selection Logic:")
print(f"    Peak F1 across all configs:   {peak_f1_configs:.4f}")
print(f"    F1 floor (95% of peak):       {f1_floor_configs:.4f}")
print(f"    Configs above F1 floor:       {len(eligible_configs)} of {len(results)}")
for k in eligible_configs:
    marker = " ◄ SELECTED" if k == best_strategy else ""
    print(f"      {k:<22s} F1={results[k]['f1']:.4f}  Recall={results[k]['recall']:.4f}{marker}")
print(f"\n  ► Best config: {best_strategy}")
print(f"    AUC={results[best_strategy]['auc']:.4f} F1={results[best_strategy]['f1']:.4f} Recall={results[best_strategy]['recall']:.4f}")

# ═══ Threshold Tuning: Best recall among thresholds with F1 within 95% of peak ═══
print(f"\n── Fine-Grained Threshold Tuning ──")
best_prob = results[best_strategy]["y_prob"]

# Collect all threshold results
threshold_results = []
for t in np.arange(0.15, 0.70, 0.025):
    yp = (best_prob >= t).astype(int)
    f1 = f1_score(y_test, yp)
    p = precision_score(y_test, yp, zero_division=0)
    r = recall_score(y_test, yp)
    cm = confusion_matrix(y_test, yp)
    threshold_results.append({"thresh": round(t, 3), "f1": f1, "prec": p, "recall": r,
                               "tp": cm[1,1], "fp": cm[0,1], "fn": cm[1,0]})

# Find peak F1, set floor at 95%, pick highest recall within floor
peak_f1_thresh = max(t["f1"] for t in threshold_results)
f1_floor_thresh = peak_f1_thresh * 0.95

eligible_thresholds = [t for t in threshold_results if t["f1"] >= f1_floor_thresh]
best_entry = max(eligible_thresholds, key=lambda t: t["recall"])
best_thresh = best_entry["thresh"]
best_f1 = best_entry["f1"]

print(f"\n  {'Thresh':>7s} │ {'F1':>7s} {'Prec':>7s} {'Recall':>7s} │ {'TP':>8s} {'FP':>8s} {'FN':>8s} │ {'Status':>12s}")
print("  " + "─" * 75)
for t in threshold_results:
    in_range = t["f1"] >= f1_floor_thresh
    marker = " ◄ SELECTED" if t["thresh"] == best_thresh else ""
    status = "✓ Eligible" if in_range else ""
    print(f"  {t['thresh']:>7.3f} │ {t['f1']:>6.4f} {t['prec']:>6.4f} {t['recall']:>6.4f} │ {t['tp']:>8,} {t['fp']:>8,} {t['fn']:>8,} │ {status:>12s}{marker}")

print(f"\n  Threshold Selection Logic:")
print(f"    Peak F1 across all thresholds: {peak_f1_thresh:.4f}")
print(f"    F1 floor (95% of peak):        {f1_floor_thresh:.4f}")
print(f"    Eligible thresholds:            {len(eligible_thresholds)} of {len(threshold_results)}")
print(f"    ► Selected: threshold={best_thresh} (F1={best_f1:.4f}, Recall={best_entry['recall']:.4f})")
print(f"    ► F1 drop from peak:           {peak_f1_thresh - best_f1:.4f} ({(peak_f1_thresh - best_f1)/peak_f1_thresh*100:.1f}%)")

# Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
strats = list(results.keys()); metrics_list = ["auc","f1","precision","recall"]
x = np.arange(len(strats)); width = 0.2
for i,m in enumerate(metrics_list):
    axes[0].bar(x+i*width, [results[s][m] for s in strats], width, label=m.upper())
axes[0].set_xticks(x+width*1.5); axes[0].set_xticklabels(strats, rotation=30, ha="right", fontsize=7)
axes[0].set_title("Config Comparison", fontweight="bold"); axes[0].legend(fontsize=8); axes[0].set_ylim(0,1.1)

thresholds_plot = np.arange(0.1, 0.9, 0.02)
f1s_plot = [f1_score(y_test, (best_prob>=t).astype(int)) for t in thresholds_plot]
recs_plot = [recall_score(y_test, (best_prob>=t).astype(int)) for t in thresholds_plot]
precs_plot = [precision_score(y_test, (best_prob>=t).astype(int), zero_division=0) for t in thresholds_plot]
axes[1].plot(thresholds_plot, f1s_plot, "b-", lw=2, label="F1")
axes[1].plot(thresholds_plot, recs_plot, "r--", label="Recall")
axes[1].plot(thresholds_plot, precs_plot, "g--", label="Precision")
axes[1].axhline(f1_floor_thresh, color="blue", linestyle=":", alpha=0.5, label=f"F1 floor ({f1_floor_thresh:.3f})")
axes[1].axvline(best_thresh, color="k", linestyle=":", alpha=0.5, label=f"Selected ({best_thresh})")
axes[1].set_title("Threshold Tuning", fontweight="bold"); axes[1].set_xlabel("Threshold"); axes[1].legend(fontsize=8)

axes[2].hist(best_prob[y_test==0], bins=50, alpha=0.6, label="Clean", color="#2ecc71", density=True)
axes[2].hist(best_prob[y_test==1], bins=50, alpha=0.6, label="AML", color="#e74c3c", density=True)
axes[2].axvline(best_thresh, color="k", linestyle="--", label=f"Threshold={best_thresh}")
axes[2].set_title("Score Distribution", fontweight="bold"); axes[2].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_phase1_tuning.png"), bbox_inches="tight"); plt.show()

CLASS IMBALANCE + HYPERPARAMETER TUNING + TYPOLOGY-AWARE SPLIT

── Typology-Aware Stratified Split ──
  Train: 267,100 | Test: 66,775 | Imbalance: 1.58:1

  Typology                                    Train     Test   Status
  ────────────────────────────────────────────────────────────────────
  Charity Abuse                              68,968   17,242 ✓
  Circular Transaction Loop                     758      190 ✓
  Clean                                     163,384   40,846 ✓
  Funnel Account Network                     22,522    5,631 ✓
  High-Risk Corridor Transfer                 1,961      490 ✓
  Money Mule Network                            767      192 ✓
  Pass-Through Transit Hub                    1,785      446 ✓
  Rapid Multi-Hop Layering                      803      200 ✓
  Structuring (Smurfing)                      1,666      417 ✓
  Third-Party Payment Web                     3,937      984 ✓
  Underground Banking (Hawala)                  549      137 ✓

── Hyperpa

KeyboardInterrupt: 

## Updated Code of section 10

In [15]:
print("=" * 90)
print("FEATURE INTERACTIONS + MODEL TRAINING + BANK-OPERATIONAL THRESHOLD")
print("=" * 90)

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

# ═══════════════════════════════════════════════════════════════
# STEP 1: Prepare Data + Typology-Aware Split
# ═══════════════════════════════════════════════════════════════
print("\n── Step 1: Data Preparation & Typology-Aware Split ──")

X = df_ml[features_final].copy()
for c in X.columns: X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)
y = df_ml["is_aml"].astype(int)

typ_col = "aml_typology"
df_ml["_primary_typology"] = df_ml[typ_col].astype(str).apply(
    lambda x: x.split("; ")[0].strip() if x and x not in ("nan", "", "None") else "Clean"
)
df_ml["_strat_key"] = df_ml["_primary_typology"]
typ_counts = df_ml["_strat_key"].value_counts()
rare = typ_counts[typ_counts < 50].index.tolist()
if rare:
    df_ml.loc[df_ml["_strat_key"].isin(rare), "_strat_key"] = "Rare_Typology"
    print(f"  Merged {len(rare)} rare typologies for stratification: {rare}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=df_ml["_strat_key"]
)
train_indices = X_train.index; test_indices = X_test.index

n_pos = (y_train == 1).sum(); n_neg = (y_train == 0).sum()
imbalance_ratio = n_neg / max(n_pos, 1)
print(f"  Train: {len(X_train):,} | Test: {len(X_test):,} | Imbalance: {imbalance_ratio:.2f}:1")

print(f"\n  {'Typology':<40s} {'Train':>8s} {'Test':>8s} {'Status':>8s}")
print(f"  {'─'*68}")
train_typs = df_ml.loc[train_indices, "_primary_typology"]
test_typs = df_ml.loc[test_indices, "_primary_typology"]
for typ in sorted(set(train_typs.unique()) | set(test_typs.unique())):
    tr = (train_typs == typ).sum(); te = (test_typs == typ).sum()
    status = "✓" if tr > 0 and te > 0 else "⚠ MISSING"
    print(f"  {typ:<40s} {tr:>8,} {te:>8,} {status}")

# ═══════════════════════════════════════════════════════════════
# STEP 2: Feature Interactions
# ═══════════════════════════════════════════════════════════════
print(f"\n── Step 2: Feature Interactions ──")

interaction_count = 0

# Velocity × Account Age: New accounts with high activity = funnel/mule
if "sender_acct_txn_count_30d" in X_train.columns:
    acct_age_col = next((c for c in df_ml.columns if "account_age" in c.lower() or "acct_open_days" in c.lower() or "account_wallet_opening" in c.lower()), None)
    if acct_age_col:
        age_train = pd.to_numeric(df_ml.loc[train_indices, acct_age_col], errors="coerce").fillna(365)
        age_test = pd.to_numeric(df_ml.loc[test_indices, acct_age_col], errors="coerce").fillna(365)
        X_train["_interact_velocity_x_new_acct"] = X_train["sender_acct_txn_count_30d"].values / np.clip(age_train.values, 1, 9999)
        X_test["_interact_velocity_x_new_acct"] = X_test["sender_acct_txn_count_30d"].values / np.clip(age_test.values, 1, 9999)
        features_final.append("_interact_velocity_x_new_acct")
        interaction_count += 1
        print(f"  Added: velocity_x_new_acct (high velocity on new accounts)")

# Amount × Counterparty Spread: Large amounts to many people = layering
if "sender_acct_unique_counterparties_7d" in X_train.columns and "transaction_amount" in X_train.columns:
    X_train["_interact_amt_x_spread"] = X_train["transaction_amount"] * np.log1p(X_train["sender_acct_unique_counterparties_7d"])
    X_test["_interact_amt_x_spread"] = X_test["transaction_amount"] * np.log1p(X_test["sender_acct_unique_counterparties_7d"])
    features_final.append("_interact_amt_x_spread")
    interaction_count += 1
    print(f"  Added: amt_x_spread (large amounts to many counterparties)")

# Inflow/Outflow Asymmetry: Accounts that receive and immediately forward = pass-through
if "sender_acct_inflow_amt_24h" in X_train.columns and "sender_acct_outflow_amt_24h" in X_train.columns:
    X_train["_interact_passthrough_ratio"] = X_train["sender_acct_outflow_amt_24h"] / np.clip(X_train["sender_acct_inflow_amt_24h"], 1, None)
    X_test["_interact_passthrough_ratio"] = X_test["sender_acct_outflow_amt_24h"] / np.clip(X_test["sender_acct_inflow_amt_24h"], 1, None)
    features_final.append("_interact_passthrough_ratio")
    interaction_count += 1
    print(f"  Added: passthrough_ratio (outflow/inflow ratio)")

# Rule Density: Many rules firing on small amounts = structuring
rule_cols_interact = [c for c in X_train.columns if c.startswith("rule_") and c != "rule_score"]
if rule_cols_interact and "transaction_amount" in X_train.columns:
    rule_sum_train = X_train[rule_cols_interact].sum(axis=1)
    rule_sum_test = X_test[rule_cols_interact].sum(axis=1)
    X_train["_interact_rules_per_lakh"] = rule_sum_train / np.clip(X_train["transaction_amount"] / 100000, 0.01, None)
    X_test["_interact_rules_per_lakh"] = rule_sum_test / np.clip(X_test["transaction_amount"] / 100000, 0.01, None)
    features_final.append("_interact_rules_per_lakh")
    interaction_count += 1
    print(f"  Added: rules_per_lakh (rule density relative to amount)")

# Balance Drain Speed: High % balance moved + high velocity = mule draining
if "sender_pct_balance_moved" in X_train.columns and "sender_acct_txn_count_24h" in X_train.columns:
    X_train["_interact_drain_speed"] = X_train["sender_pct_balance_moved"] * X_train["sender_acct_txn_count_24h"]
    X_test["_interact_drain_speed"] = X_test["sender_pct_balance_moved"] * X_test["sender_acct_txn_count_24h"]
    features_final.append("_interact_drain_speed")
    interaction_count += 1
    print(f"  Added: drain_speed (balance drain rate)")

# Cross-Border × Amount: International + high value = corridor transfer
if "ip_flag_cross_border" in X_train.columns and "transaction_amount" in X_train.columns:
    X_train["_interact_xborder_amt"] = X_train["ip_flag_cross_border"] * X_train["transaction_amount"]
    X_test["_interact_xborder_amt"] = X_test["ip_flag_cross_border"] * X_test["transaction_amount"]
    features_final.append("_interact_xborder_amt")
    interaction_count += 1
    print(f"  Added: xborder_amt (cross-border high value)")

# IP Risk × Velocity: Risky device with burst activity = account takeover
if "ip_risk_score" in X_train.columns and "sender_acct_txn_count_1h" in X_train.columns:
    X_train["_interact_risky_device_burst"] = X_train["ip_risk_score"] * X_train["sender_acct_txn_count_1h"]
    X_test["_interact_risky_device_burst"] = X_test["ip_risk_score"] * X_test["sender_acct_txn_count_1h"]
    features_final.append("_interact_risky_device_burst")
    interaction_count += 1
    print(f"  Added: risky_device_burst (VPN/emulator + burst)")

# Receiver Concentration: Many senders + high inflow = funnel account
if "receiver_acct_unique_senders_7d" in X_train.columns and "receiver_acct_inflow_amt_30d" in X_train.columns:
    X_train["_interact_funnel_signal"] = X_train["receiver_acct_unique_senders_7d"] * np.log1p(X_train["receiver_acct_inflow_amt_30d"])
    X_test["_interact_funnel_signal"] = X_test["receiver_acct_unique_senders_7d"] * np.log1p(X_test["receiver_acct_inflow_amt_30d"])
    features_final.append("_interact_funnel_signal")
    interaction_count += 1
    print(f"  Added: funnel_signal (many senders + high inflow)")

features_final = list(dict.fromkeys(features_final))
print(f"\n  Interaction features added: {interaction_count}")
print(f"  Total features: {len(features_final)}")

# ═══════════════════════════════════════════════════════════════
# STEP 3: SMOTE
# ═══════════════════════════════════════════════════════════════
print(f"\n── Step 3: SMOTE Oversampling ──")
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    print(f"  Before: {len(X_train):,} (AML={n_pos:,}) → After: {len(X_train_smote):,} (AML={(y_train_smote==1).sum():,})")
except ImportError:
    HAS_SMOTE = False; X_train_smote = X_train; y_train_smote = y_train
    print("  imblearn not installed — skipping SMOTE")

# ═══════════════════════════════════════════════════════════════
# STEP 4: Hyperparameter Tuning (12 configurations)
# ═══════════════════════════════════════════════════════════════
print(f"\n── Step 4: Hyperparameter Tuning (12 configurations) ──\n")

tuning_configs = [
    {"name":"Baseline",          "nl":63,  "md":8,  "mc":50,  "lr":0.05, "ra":0,    "rl":0,   "sub":0.8, "col":0.8, "data":"orig",  "spw":imbalance_ratio},
    {"name":"Deep+Reg",          "nl":127, "md":10, "mc":30,  "lr":0.03, "ra":0.1,  "rl":1.0, "sub":0.8, "col":0.8, "data":"orig",  "spw":imbalance_ratio},
    {"name":"Slow+Deep+L2",      "nl":127, "md":10, "mc":20,  "lr":0.01, "ra":0.1,  "rl":2.0, "sub":0.7, "col":0.7, "data":"orig",  "spw":imbalance_ratio},
    {"name":"Wide+Shallow",      "nl":255, "md":6,  "mc":100, "lr":0.05, "ra":0.1,  "rl":1.0, "sub":0.7, "col":0.8, "data":"orig",  "spw":imbalance_ratio},
    {"name":"SMOTE+Balanced",    "nl":63,  "md":8,  "mc":50,  "lr":0.05, "ra":0,    "rl":0,   "sub":0.8, "col":0.8, "data":"smote", "spw":1.0},
    {"name":"SMOTE+Deep+Reg",    "nl":127, "md":10, "mc":30,  "lr":0.03, "ra":0.1,  "rl":1.0, "sub":0.8, "col":0.8, "data":"smote", "spw":1.0},
    {"name":"SMOTE+HalfWt",      "nl":127, "md":10, "mc":30,  "lr":0.03, "ra":0.1,  "rl":1.0, "sub":0.8, "col":0.8, "data":"smote", "spw":imbalance_ratio*0.5},
    {"name":"HeavyReg+Slow",     "nl":31,  "md":6,  "mc":100, "lr":0.01, "ra":0.5,  "rl":5.0, "sub":0.6, "col":0.7, "data":"orig",  "spw":imbalance_ratio},
    {"name":"XDeep+SlowLR",      "nl":200, "md":12, "mc":15,  "lr":0.008,"ra":0.05, "rl":0.5, "sub":0.75,"col":0.75,"data":"orig",  "spw":imbalance_ratio},
    {"name":"SMOTE+XDeep",       "nl":200, "md":12, "mc":15,  "lr":0.008,"ra":0.05, "rl":0.5, "sub":0.75,"col":0.75,"data":"smote", "spw":1.0},
    {"name":"HighLeaves+Reg",    "nl":300, "md":8,  "mc":50,  "lr":0.01, "ra":0.2,  "rl":2.0, "sub":0.7, "col":0.7, "data":"orig",  "spw":imbalance_ratio},
    {"name":"SMOTE+HighLeaves",  "nl":300, "md":8,  "mc":50,  "lr":0.01, "ra":0.2,  "rl":2.0, "sub":0.7, "col":0.7, "data":"smote", "spw":1.0},
]

print(f"  {'Config':<22s} │ {'AUC':>7s} {'F1':>7s} {'Prec':>7s} {'Recall':>7s} │ {'Rounds':>6s} {'Thresh':>6s}")
print("  " + "─" * 78)

results = {}

for cfg in tuning_configs:
    params = {"objective":"binary","metric":"auc","num_leaves":cfg["nl"],"max_depth":cfg["md"],
              "min_child_samples":cfg["mc"],"learning_rate":cfg["lr"],"reg_alpha":cfg["ra"],
              "reg_lambda":cfg["rl"],"subsample":cfg["sub"],"colsample_bytree":cfg["col"],
              "scale_pos_weight":cfg["spw"],"verbosity":-1,"random_state":42,"n_jobs":-1}

    Xtr = X_train_smote if cfg["data"]=="smote" else X_train
    ytr = y_train_smote if cfg["data"]=="smote" else y_train

    ds = lgb.Dataset(Xtr, label=ytr)
    val = lgb.Dataset(X_test, label=y_test, reference=ds)
    mdl = lgb.train(params, ds, num_boost_round=1000,
                    valid_sets=[val], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

    prob = mdl.predict(X_test); auc_v = roc_auc_score(y_test, prob)
    bt=0.5; bf1=0
    for t in np.arange(0.25, 0.75, 0.02):
        f = f1_score(y_test, (prob>=t).astype(int))
        if f > bf1: bf1=f; bt=round(t,2)

    pred = (prob>=bt).astype(int)
    f1_v=f1_score(y_test,pred); p_v=precision_score(y_test,pred); r_v=recall_score(y_test,pred)
    cm=confusion_matrix(y_test,pred)

    results[cfg["name"]] = {"auc":auc_v,"f1":f1_v,"precision":p_v,"recall":r_v,
                            "model":mdl,"y_prob":prob,"threshold":bt,
                            "tp":cm[1,1],"fp":cm[0,1],"fn":cm[1,0],"tn":cm[0,0],
                            "config":cfg,"rounds":mdl.best_iteration if hasattr(mdl,"best_iteration") else 0}

    print(f"  {cfg['name']:<22s} │ {auc_v:>6.4f} {f1_v:>6.4f} {p_v:>6.4f} {r_v:>6.4f} │ {results[cfg['name']]['rounds']:>6} {bt:>5.2f}")

# ═══════════════════════════════════════════════════════════════
# STEP 5: Config Selection — Best recall within F1 safety zone
# ═══════════════════════════════════════════════════════════════
print(f"\n── Step 5: Config Selection ──")

peak_f1_configs = max(v["f1"] for v in results.values())
f1_floor_configs = peak_f1_configs * 0.95

eligible_configs = {k: v for k, v in results.items() if v["f1"] >= f1_floor_configs}
best_strategy = max(eligible_configs, key=lambda k: eligible_configs[k]["recall"])

print(f"\n  Peak F1 across configs: {peak_f1_configs:.4f}")
print(f"  F1 safety zone (95%):  {f1_floor_configs:.4f}")
print(f"  Configs in safety zone: {len(eligible_configs)} of {len(results)}")
for k in eligible_configs:
    marker = " ◄ SELECTED" if k == best_strategy else ""
    print(f"    {k:<22s} F1={results[k]['f1']:.4f}  Recall={results[k]['recall']:.4f}{marker}")
print(f"\n  ► Selected: {best_strategy}")

# ═══════════════════════════════════════════════════════════════
# STEP 6: Bank-Operational Threshold Calibration
# ═══════════════════════════════════════════════════════════════
print(f"\n── Step 6: Threshold Calibration (Bank-Operational) ──")

best_prob = results[best_strategy]["y_prob"]
total_aml_test = (y_test == 1).sum()
total_clean_test = (y_test == 0).sum()

# Collect all threshold results with bank-operational metrics
threshold_results = []
for t in np.arange(0.20, 0.70, 0.025):
    yp = (best_prob >= t).astype(int)
    cm = confusion_matrix(y_test, yp)
    tp = cm[1,1]; fp = cm[0,1]; fn = cm[1,0]; tn = cm[0,0]
    
    total_alerts = tp + fp
    alert_quality = tp / max(total_alerts, 1)  # precision = what % of alerts are real
    missed_aml = fn
    detection_rate = tp / max(total_aml_test, 1)  # recall
    false_alarm_rate = fp / max(total_clean_test, 1)
    analyst_hours_monthly = total_alerts * 5 * 0.25  # 5x scale for monthly, 15 min each
    
    threshold_results.append({
        "thresh": round(t, 3),
        "total_alerts": total_alerts, "real_aml": tp, "false_alarms": fp,
        "missed_aml": missed_aml, "correctly_cleared": tn,
        "alert_quality_pct": round(alert_quality * 100, 1),
        "detection_rate_pct": round(detection_rate * 100, 1),
        "false_alarm_rate_pct": round(false_alarm_rate * 100, 1),
        "analyst_hours": round(analyst_hours_monthly),
        "f1": f1_score(y_test, yp),
    })

# Find peak F1 for threshold safety zone
peak_f1_thresh = max(t["f1"] for t in threshold_results)
f1_floor_thresh = peak_f1_thresh * 0.95

print(f"\n  Reading the table:")
print(f"    Total Alerts  = transactions sent to analyst team for review")
print(f"    Real AML      = alerts that are actually money laundering (true catches)")
print(f"    False Alarms  = clean transactions incorrectly flagged (wasted reviews)")
print(f"    Missed AML    = money laundering that slipped through undetected (regulatory risk)")
print(f"    Alert Quality = what % of alerts are genuine (higher = less wasted time)")
print(f"    Detection     = what % of all AML is caught (higher = safer)")
print(f"    Analyst Hrs   = estimated monthly review hours at 15 min/alert")

print(f"\n  {'Thresh':>7s} │ {'Alerts':>8s} {'Real AML':>9s} {'False':>7s} {'Missed':>8s} │ {'Quality':>8s} {'Detection':>10s} │ {'Hrs/Mo':>8s} │ {'Zone':>10s}")
print("  " + "─" * 100)

best_thresh = 0.5  # fallback
best_score = -999999

for t in threshold_results:
    in_zone = t["f1"] >= f1_floor_thresh
    
    # Bank-operational scoring:
    # Minimize missed AML (most important — regulatory risk)
    # But only consider thresholds where alert quality >= 65% (analyst feasibility)
    # And F1 stays in safety zone (model isn't degrading)
    
    quality_ok = t["alert_quality_pct"] >= 65
    
    if in_zone and quality_ok:
        # Score = negative missed AML (fewer missed = better)
        op_score = -t["missed_aml"]
        zone_label = "✓ Eligible"
    elif in_zone and not quality_ok:
        zone_label = "● F1 OK, quality low"
        op_score = -999999
    else:
        zone_label = ""
        op_score = -999999
    
    marker = ""
    if op_score > best_score:
        best_score = op_score
        best_thresh = t["thresh"]
        marker = " ◄"
    
    print(f"  {t['thresh']:>7.3f} │ {t['total_alerts']:>8,} {t['real_aml']:>9,} {t['false_alarms']:>7,} {t['missed_aml']:>8,} │ {t['alert_quality_pct']:>7.1f}% {t['detection_rate_pct']:>9.1f}% │ {t['analyst_hours']:>8,} │ {zone_label:>10s}{marker}")

# Get final entry details
final_entry = next(t for t in threshold_results if t["thresh"] == best_thresh)

print(f"\n  ┌──────────────────────────────────────────────────────────────────────────────┐")
print(f"  │  SELECTED OPERATING POINT                                                    │")
print(f"  │  Threshold:       {best_thresh}                                                       │")
print(f"  │  Alert Quality:   {final_entry['alert_quality_pct']}% (of alerts sent to analysts, this % are real)    │")
print(f"  │  Detection Rate:  {final_entry['detection_rate_pct']}% (of all AML, this % is caught)                  │")
print(f"  │  Monthly Alerts:  ~{final_entry['total_alerts'] * 5:,} (at 15 min each = ~{final_entry['analyst_hours']:,} analyst hours)      │")
print(f"  │  Missed AML:      {final_entry['missed_aml']:,} per test batch ({final_entry['missed_aml'] * 5:,} estimated monthly)          │")
print(f"  └──────────────────────────────────────────────────────────────────────────────┘")

# ═══════════════════════════════════════════════════════════════
# STEP 7: Plots
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f"Model Selection & Threshold Calibration — {best_strategy}", fontsize=14, fontweight="bold")

# Plot 1: Config comparison
strats = list(results.keys()); metrics_list = ["auc","f1","precision","recall"]
x_pos = np.arange(len(strats)); width = 0.2
for i, m in enumerate(metrics_list):
    axes[0,0].bar(x_pos+i*width, [results[s][m] for s in strats], width, label=m.upper())
axes[0,0].set_xticks(x_pos+width*1.5); axes[0,0].set_xticklabels(strats, rotation=35, ha="right", fontsize=6)
axes[0,0].set_title("Model Configuration Comparison", fontweight="bold"); axes[0,0].legend(fontsize=7); axes[0,0].set_ylim(0, 1.1)

# Plot 2: Bank-operational view — Alerts vs Missed AML vs Threshold
threshs = [t["thresh"] for t in threshold_results]
alerts = [t["total_alerts"] for t in threshold_results]
missed = [t["missed_aml"] for t in threshold_results]
quality = [t["alert_quality_pct"] for t in threshold_results]

ax2a = axes[0,1]
ax2b = ax2a.twinx()
ax2a.plot(threshs, alerts, "b-", lw=2, label="Total Alerts (left)")
ax2a.plot(threshs, missed, "r-", lw=2, label="Missed AML (left)")
ax2b.plot(threshs, quality, "g--", lw=2, label="Alert Quality % (right)")
ax2a.axvline(best_thresh, color="k", linestyle=":", alpha=0.5)
ax2a.axhline(0, color="gray", linestyle="-", alpha=0.2)
ax2b.axhline(65, color="green", linestyle=":", alpha=0.5)
ax2a.set_xlabel("Threshold"); ax2a.set_ylabel("Count"); ax2b.set_ylabel("Quality %")
ax2a.set_title("Bank Operations: Alerts vs Missed AML", fontweight="bold")
lines1, labels1 = ax2a.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2a.legend(lines1+lines2, labels1+labels2, fontsize=8, loc="center right")

# Plot 3: Score distribution
axes[1,0].hist(best_prob[y_test==0], bins=50, alpha=0.6, label="Clean", color="#2ecc71", density=True)
axes[1,0].hist(best_prob[y_test==1], bins=50, alpha=0.6, label="AML", color="#e74c3c", density=True)
axes[1,0].axvline(best_thresh, color="k", linestyle="--", lw=2, label=f"Threshold={best_thresh}")
axes[1,0].set_title("Score Distribution: Clean vs AML", fontweight="bold"); axes[1,0].legend()
axes[1,0].set_xlabel("Model Predicted Probability")

# Plot 4: Detection Rate vs Alert Quality (operational tradeoff)
det_rates = [t["detection_rate_pct"] for t in threshold_results]
axes[1,1].plot(det_rates, quality, "b-o", lw=2, markersize=4)
# Mark selected point
sel_det = final_entry["detection_rate_pct"]
sel_qual = final_entry["alert_quality_pct"]
axes[1,1].plot(sel_det, sel_qual, "r*", markersize=15, label=f"Selected ({sel_det}%, {sel_qual}%)")
axes[1,1].axhline(65, color="green", linestyle=":", alpha=0.5, label="Min Quality (65%)")
axes[1,1].set_xlabel("Detection Rate (% of AML caught)")
axes[1,1].set_ylabel("Alert Quality (% of alerts that are real)")
axes[1,1].set_title("Operational Tradeoff: Detection vs Quality", fontweight="bold")
axes[1,1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_phase1_tuning.png"), bbox_inches="tight")
plt.show()
print(f"\n  Saved: {OUTPUT_DIR}/06_phase1_tuning.png")

FEATURE INTERACTIONS + MODEL TRAINING + BANK-OPERATIONAL THRESHOLD

── Step 1: Data Preparation & Typology-Aware Split ──
  Train: 267,100 | Test: 66,775 | Imbalance: 1.58:1

  Typology                                    Train     Test   Status
  ────────────────────────────────────────────────────────────────────
  Charity Abuse                              68,968   17,242 ✓
  Circular Transaction Loop                     758      190 ✓
  Clean                                     163,384   40,846 ✓
  Funnel Account Network                     22,522    5,631 ✓
  High-Risk Corridor Transfer                 1,961      490 ✓
  Money Mule Network                            767      192 ✓
  Pass-Through Transit Hub                    1,785      446 ✓
  Rapid Multi-Hop Layering                      803      200 ✓
  Structuring (Smurfing)                      1,666      417 ✓
  Third-Party Payment Web                     3,937      984 ✓
  Underground Banking (Hawala)                  549   

## 11 — Final Baseline Model (Best Imbalance Strategy)


In [16]:
print(X_test.shape)
print(X_train.shape)

(66775, 141)
(267100, 141)


In [30]:
best_thresh, best_strategy

#best_thresh = 0.45

(0.5, 'Class Weights')

In [31]:
print("=" * 90)
print(f"FINAL BASELINE MODEL — {best_strategy} + Threshold={best_thresh}")
print("=" * 90)

if HAS_LGB:
    from sklearn.metrics import classification_report, roc_curve, precision_recall_curve, average_precision_score
    
    final_model = results[best_strategy]["model"]
    y_prob_final = results[best_strategy]["y_prob"]
    y_pred_final = (y_prob_final >= best_thresh).astype(int)
    
    auc = roc_auc_score(y_test, y_prob_final)
    avg_prec = average_precision_score(y_test, y_prob_final)
    
    print(f"\n  AUC-ROC:           {auc:.4f}")
    print(f"  Average Precision: {avg_prec:.4f}")
    print(f"  Threshold:         {best_thresh}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred_final, target_names=["Clean", "AML"], digits=4))
    
    cm = confusion_matrix(y_test, y_pred_final)
    print(f"  Confusion Matrix:")
    print(f"    {'':>15s} {'Pred Clean':>12s} {'Pred AML':>12s}")
    print(f"    {'Actual Clean':<15s} {cm[0,0]:>12,} {cm[0,1]:>12,}")
    print(f"    {'Actual AML':<15s} {cm[1,0]:>12,} {cm[1,1]:>12,}")
    
    # Per-typology recall
    if "aml_typology" in df.columns:
        print(f"\n  ── Per-Typology Detection Rate ──")
        test_df = df.iloc[X_test.index].copy()
        test_df["_pred_prob"] = y_prob_final
        test_df["_pred"] = y_pred_final
        typ_col = "aml_typology"
        all_typs = set()
        for t in test_df[typ_col].dropna():
            for part in str(t).split("; "):
                if part.strip(): all_typs.add(part.strip())
        
        print(f"    {'Typology':<40s} {'Total':>7s} {'Caught':>7s} {'Recall':>7s} {'Avg Prob':>9s} {'Status':>8s}")
        print(f"    {'─'*83}")
        for typ in sorted(all_typs):
            mask = test_df[typ_col].astype(str).str.contains(typ, na=False)
            cnt = mask.sum()
            if cnt == 0: continue
            caught = test_df.loc[mask, "_pred"].sum()
            recall = caught / cnt * 100
            avg_p = test_df.loc[mask, "_pred_prob"].mean()
            status = "✓ GOOD" if recall > 80 else ("⚡ CHECK" if recall > 50 else "⚠ LOW")
            print(f"    {typ:<40s} {cnt:>7,} {caught:>7,} {recall:>6.1f}% {avg_p:>8.3f} {status}")
    
    # Feature importance
    imp = pd.DataFrame({"feature": features_final,
                         "gain": final_model.feature_importance(importance_type="gain")}).sort_values("gain", ascending=False)
    
    print(f"\n  ── Top 20 Features by Importance ──")
    print(f"    {'Rank':<5s} {'Feature':<55s} {'Gain':>12s} {'Category':>15s}")
    print(f"    {'─'*90}")
    for i, (_, row) in enumerate(imp.head(20).iterrows(), 1):
        cat = "RULE" if row["feature"].startswith("rule_") else ("ENCODED" if row["feature"].endswith("_enc") else "GRAPH/VEL")
        print(f"    {i:<5d} {row['feature']:<55s} {row['gain']:>12.1f} {cat:>15s}")
    
    # Plots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"Final Model Results — {best_strategy} (threshold={best_thresh})", fontsize=14, fontweight="bold")
    
    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob_final)
    axes[0,0].plot(fpr, tpr, "b-", lw=2, label=f"AUC={auc:.4f}")
    axes[0,0].plot([0,1],[0,1],"k--",alpha=0.3); axes[0,0].set_title("ROC Curve"); axes[0,0].legend()
    
    # PR
    prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob_final)
    axes[0,1].plot(rec_c, prec_c, "r-", lw=2, label=f"AP={avg_prec:.4f}")
    axes[0,1].set_title("Precision-Recall Curve"); axes[0,1].legend()
    
    # Confusion matrix heatmap
    sns.heatmap(cm, annot=True, fmt=",", cmap="Blues", ax=axes[1,0],
                xticklabels=["Pred Clean","Pred AML"], yticklabels=["Actual Clean","Actual AML"])
    axes[1,0].set_title("Confusion Matrix")
    
    # Top 15 importance
    top15 = imp.head(15).sort_values("gain")
    colors = ["#e74c3c" if f.startswith("rule_") else "#3498db" if f.endswith("_enc") else "#2ecc71" for f in top15["feature"]]
    axes[1,1].barh(top15["feature"], top15["gain"], color=colors)
    axes[1,1].set_title("Top 15 Feature Importance (Red=Rules, Blue=Categorical, Green=Graph)")
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "07_final_model_results.png"), bbox_inches="tight")
    plt.show()
    
    # Save model + metadata
    final_model.save_model(os.path.join(OUTPUT_DIR, "final_lgb_model.txt"))
    
    X_train.to_parquet(os.path.join(OUTPUT_DIR, "X_train.parquet"), index=False)
    X_test.to_parquet(os.path.join(OUTPUT_DIR, "X_test.parquet"), index=False)
    y_train.to_frame().to_parquet(os.path.join(OUTPUT_DIR, "y_train.parquet"), index=False)
    y_test.to_frame().to_parquet(os.path.join(OUTPUT_DIR, "y_test.parquet"), index=False)
    
    import json as _json
    metadata = {
        "features": features_final,
        "n_features": len(features_final),
        "protected_count": len(PROTECTED_FEATURES),
        "selected_graph_count": len(selected_graph_features),
        "best_strategy": best_strategy,
        "optimal_threshold": best_thresh,
        "auc_roc": auc,
        "f1_score": float(f1_score(y_test, y_pred_final)),
        "imbalance_ratio": imbalance_ratio,
        "n_train": len(X_train),
        "n_test": len(X_test),
    }
    with open(os.path.join(OUTPUT_DIR, "model_metadata.json"), "w") as f:
        _json.dump(metadata, f, indent=2)
    
    print(f"\n  Saved outputs:")
    for fn in sorted(os.listdir(OUTPUT_DIR)):
        if not os.path.isdir(os.path.join(OUTPUT_DIR, fn)):
            size = os.path.getsize(os.path.join(OUTPUT_DIR, fn)) / (1024*1024)
            print(f"    {fn:<45s} {size:>8.2f} MB")

print(f"\n{'='*90}")
print("ML PREPARATION COMPLETE")
print(f"{'='*90}")



FINAL BASELINE MODEL — Class Weights + Threshold=0.5

  AUC-ROC:           0.8660
  Average Precision: 0.8284
  Threshold:         0.5

  Classification Report:
              precision    recall  f1-score   support

       Clean     0.8348    0.8226    0.8286     40846
         AML     0.7268    0.7435    0.7350     25929

    accuracy                         0.7919     66775
   macro avg     0.7808    0.7830    0.7818     66775
weighted avg     0.7928    0.7919    0.7923     66775

  Confusion Matrix:
                      Pred Clean     Pred AML
    Actual Clean          33,598        7,248
    Actual AML             6,651       19,278

  ── Per-Typology Detection Rate ──
    Typology                                   Total  Caught  Recall  Avg Prob   Status
    ───────────────────────────────────────────────────────────────────────────────────
    Charity Abuse                             17,239  14,814   85.9%    0.762 ✓ GOOD
    Circular Transaction Loop                    233    

## 12 — Isolation Forest (Unsupervised Anomaly Detection)
Comparison model that **does not use is_aml labels** — finds anomalies based on how easy each transaction is to isolate.

**Why include this?**
- Validates whether the labels themselves are meaningful (if Isolation Forest finds the same transactions, the patterns are real anomalies)
- Production fallback: when new typologies emerge that aren't in the rules, unsupervised detection still works
- Independent second opinion to compare against LightGBM

**Key difference from LightGBM:**
- LightGBM (supervised): learns 'what AML looks like' from labeled examples
- Isolation Forest (unsupervised): learns 'what's unusual' without any labels
- Best transactions to investigate: flagged by BOTH (high confidence) | Only Isolation Forest (potential new patterns)


In [13]:
print("=" * 90)
print("ISOLATION FOREST — Unsupervised Anomaly Detection")
print("=" * 90)

from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

# ══════════════════════════════════════════════════════════════
# STEP 1: Configure Isolation Forest
# ══════════════════════════════════════════════════════════════
print("\n── Step 1: Model Configuration ──")

# contamination = expected fraction of anomalies in data
# Set this to the actual AML rate so the model knows how many to flag
contamination_rate = float(y_train.mean())
print(f"  Contamination rate: {contamination_rate:.4f} ({contamination_rate*100:.1f}% expected anomalies)")
print(f"  Number of trees: 200")
print(f"  Max samples per tree: 256 (default)")
print(f"  Random state: 42")

iso_model = IsolationForest(
    n_estimators=200,
    max_samples=256,
    contamination=contamination_rate,
    max_features=1.0,
    bootstrap=False,
    n_jobs=-1,
    random_state=42,
    verbose=0
)

# ══════════════════════════════════════════════════════════════
# STEP 2: Train (note: doesn't use y_train — purely unsupervised)
# ══════════════════════════════════════════════════════════════
print("\n── Step 2: Training Isolation Forest ──")
print("  Note: This is UNSUPERVISED — y_train is NOT used during training")
print("  The model learns what 'normal' looks like and flags outliers")

import time
t0 = time.time()
iso_model.fit(X_train)
train_time = time.time() - t0
print(f"  Training time: {train_time:.1f}s")

# ══════════════════════════════════════════════════════════════
# STEP 3: Predictions
# ══════════════════════════════════════════════════════════════
print("\n── Step 3: Generate Predictions ──")

# Anomaly scores: lower = more anomalous (we negate so higher = more anomalous, matching LightGBM convention)
iso_scores_raw = iso_model.score_samples(X_test)  # negative scores, higher = more normal
iso_anomaly_score = -iso_scores_raw  # flip so higher = more suspicious

# Normalize to 0-1 range for comparability with LightGBM
iso_score_normalized = (iso_anomaly_score - iso_anomaly_score.min()) / (iso_anomaly_score.max() - iso_anomaly_score.min())

# Binary prediction (1 = anomaly/AML, 0 = normal/clean)
iso_pred_default = iso_model.predict(X_test)  # returns -1 (anomaly) or 1 (normal)
iso_pred = (iso_pred_default == -1).astype(int)  # convert to 1=anomaly, 0=normal

print(f"  Predicted anomalies: {iso_pred.sum():,} ({iso_pred.mean()*100:.1f}% of test set)")
print(f"  Score range: [{iso_anomaly_score.min():.4f}, {iso_anomaly_score.max():.4f}]")

# ══════════════════════════════════════════════════════════════
# STEP 4: Evaluate Against True Labels (is_aml)
# ══════════════════════════════════════════════════════════════
print("\n── Step 4: Evaluation Against Ground Truth ──")

iso_auc = roc_auc_score(y_test, iso_anomaly_score)
iso_f1 = f1_score(y_test, iso_pred)
iso_prec = precision_score(y_test, iso_pred)
iso_rec = recall_score(y_test, iso_pred)

cm_iso = confusion_matrix(y_test, iso_pred)
tp_i = cm_iso[1,1]; fp_i = cm_iso[0,1]; fn_i = cm_iso[1,0]; tn_i = cm_iso[0,0]

print(f"\n  Isolation Forest Performance:")
print(f"    AUC-ROC:    {iso_auc:.4f}")
print(f"    F1:         {iso_f1:.4f}")
print(f"    Precision:  {iso_prec:.4f}")
print(f"    Recall:     {iso_rec:.4f}")
print(f"    TP={tp_i:,}  FP={fp_i:,}  FN={fn_i:,}  TN={tn_i:,}")

# ══════════════════════════════════════════════════════════════
# STEP 5: Compare with LightGBM
# ══════════════════════════════════════════════════════════════
print("\n── Step 5: Isolation Forest vs LightGBM Head-to-Head ──")

lgb_auc = auc
lgb_f1 = f1_score(y_test, y_pred_final)
lgb_prec = precision_score(y_test, y_pred_final)
lgb_rec = recall_score(y_test, y_pred_final)
cm_lgb = confusion_matrix(y_test, y_pred_final)

print(f"\n  {'Metric':<20s} │ {'LightGBM':>12s} {'Isolation Forest':>18s} {'Winner':>10s}")
print(f"  {'─'*70}")
for metric_name, lgb_v, iso_v in [
    ("AUC-ROC", lgb_auc, iso_auc),
    ("F1 Score", lgb_f1, iso_f1),
    ("Precision", lgb_prec, iso_prec),
    ("Recall", lgb_rec, iso_rec),
]:
    winner = "LightGBM" if lgb_v > iso_v else ("IsoForest" if iso_v > lgb_v else "Tie")
    print(f"  {metric_name:<20s} │ {lgb_v:>12.4f} {iso_v:>18.4f} {winner:>10s}")

print(f"\n  {'Confusion':<20s} │ {'LightGBM':>12s} {'Isolation Forest':>18s}")
print(f"  {'─'*60}")
print(f"  {'True Positives':<20s} │ {cm_lgb[1,1]:>12,} {tp_i:>18,}")
print(f"  {'False Positives':<20s} │ {cm_lgb[0,1]:>12,} {fp_i:>18,}")
print(f"  {'False Negatives':<20s} │ {cm_lgb[1,0]:>12,} {fn_i:>18,}")
print(f"  {'True Negatives':<20s} │ {cm_lgb[0,0]:>12,} {tn_i:>18,}")

# ══════════════════════════════════════════════════════════════
# STEP 6: Per-Typology Detection by Isolation Forest
# ══════════════════════════════════════════════════════════════
print("\n── Step 6: Per-Typology Detection (Isolation Forest) ──")

if "aml_typology" in df.columns:
    test_df_iso = df.iloc[X_test.index].copy()
    test_df_iso["_iso_pred"] = iso_pred
    test_df_iso["_iso_score"] = iso_anomaly_score
    test_df_iso["_lgb_pred"] = y_pred_final
    
    typ_col = "aml_typology"
    all_typs = set()
    for t in test_df_iso[typ_col].dropna():
        for part in str(t).split("; "):
            if part.strip(): all_typs.add(part.strip())
    
    print(f"\n  {'Typology':<35s} │ {'Total':>7s} │ {'IsoF Caught':>11s} {'IsoF %':>7s} │ {'LGB Caught':>10s} {'LGB %':>7s} │ {'Both':>6s} {'IsoOnly':>8s} {'LGBOnly':>8s}")
    print(f"  {'─'*120}")
    
    for typ in sorted(all_typs):
        mask = test_df_iso[typ_col].astype(str).str.contains(typ, na=False)
        cnt = mask.sum()
        if cnt == 0: continue
        
        iso_caught = test_df_iso.loc[mask, "_iso_pred"].sum()
        lgb_caught = test_df_iso.loc[mask, "_lgb_pred"].sum()
        both = ((test_df_iso.loc[mask, "_iso_pred"] == 1) & (test_df_iso.loc[mask, "_lgb_pred"] == 1)).sum()
        iso_only = ((test_df_iso.loc[mask, "_iso_pred"] == 1) & (test_df_iso.loc[mask, "_lgb_pred"] == 0)).sum()
        lgb_only = ((test_df_iso.loc[mask, "_iso_pred"] == 0) & (test_df_iso.loc[mask, "_lgb_pred"] == 1)).sum()
        
        print(f"  {typ:<35s} │ {cnt:>7,} │ {iso_caught:>11,} {iso_caught/cnt*100:>6.1f}% │ {lgb_caught:>10,} {lgb_caught/cnt*100:>6.1f}% │ {both:>6,} {iso_only:>8,} {lgb_only:>8,}")

# ══════════════════════════════════════════════════════════════
# STEP 7: Agreement Analysis
# ══════════════════════════════════════════════════════════════
print("\n── Step 7: Model Agreement Analysis ──")

agree_both_aml = ((iso_pred == 1) & (y_pred_final == 1)).sum()
agree_both_clean = ((iso_pred == 0) & (y_pred_final == 0)).sum()
disagree_iso_only = ((iso_pred == 1) & (y_pred_final == 0)).sum()
disagree_lgb_only = ((iso_pred == 0) & (y_pred_final == 1)).sum()

total_test = len(y_test)
agreement_pct = (agree_both_aml + agree_both_clean) / total_test * 100

print(f"\n  Total test transactions: {total_test:,}")
print(f"  Both flagged AML:        {agree_both_aml:>10,} ({agree_both_aml/total_test*100:.1f}%) — HIGH CONFIDENCE alerts")
print(f"  Both said Clean:         {agree_both_clean:>10,} ({agree_both_clean/total_test*100:.1f}%) — HIGH CONFIDENCE clean")
print(f"  Only IsoForest flagged:  {disagree_iso_only:>10,} ({disagree_iso_only/total_test*100:.1f}%) — Potential new patterns")
print(f"  Only LightGBM flagged:   {disagree_lgb_only:>10,} ({disagree_lgb_only/total_test*100:.1f}%) — Known typology patterns")
print(f"  Overall agreement:       {agreement_pct:.1f}%")

# Of the "Both flagged AML" group, how many are actually AML?
mask_both_aml = (iso_pred == 1) & (y_pred_final == 1)
mask_iso_only = (iso_pred == 1) & (y_pred_final == 0)
mask_lgb_only = (iso_pred == 0) & (y_pred_final == 1)

if mask_both_aml.sum() > 0:
    both_actual_aml = y_test.values[mask_both_aml].sum()
    both_precision = both_actual_aml / mask_both_aml.sum() * 100
    print(f"\n  Precision when BOTH flag AML:    {both_actual_aml:,}/{mask_both_aml.sum():,} = {both_precision:.1f}% (HIGHEST CONFIDENCE)")

if mask_iso_only.sum() > 0:
    iso_only_actual = y_test.values[mask_iso_only].sum()
    iso_only_prec = iso_only_actual / mask_iso_only.sum() * 100
    print(f"  Precision when only IsoF flags:  {iso_only_actual:,}/{mask_iso_only.sum():,} = {iso_only_prec:.1f}%")

if mask_lgb_only.sum() > 0:
    lgb_only_actual = y_test.values[mask_lgb_only].sum()
    lgb_only_prec = lgb_only_actual / mask_lgb_only.sum() * 100
    print(f"  Precision when only LGB flags:   {lgb_only_actual:,}/{mask_lgb_only.sum():,} = {lgb_only_prec:.1f}%")

# ══════════════════════════════════════════════════════════════
# STEP 8: Visualizations
# ══════════════════════════════════════════════════════════════
print("\n── Step 8: Generating Plots ──")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Isolation Forest vs LightGBM — Comprehensive Comparison", fontsize=14, fontweight="bold")

# Plot 1: Metric comparison bar chart
metrics_names = ["AUC-ROC", "F1", "Precision", "Recall"]
lgb_vals = [lgb_auc, lgb_f1, lgb_prec, lgb_rec]
iso_vals = [iso_auc, iso_f1, iso_prec, iso_rec]
x = np.arange(len(metrics_names))
width = 0.35
axes[0,0].bar(x - width/2, lgb_vals, width, label="LightGBM", color="#3498db")
axes[0,0].bar(x + width/2, iso_vals, width, label="Isolation Forest", color="#e67e22")
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(metrics_names)
axes[0,0].set_title("Performance Metrics Comparison", fontweight="bold")
axes[0,0].set_ylim(0, 1.1); axes[0,0].legend()
for i, (l, i_v) in enumerate(zip(lgb_vals, iso_vals)):
    axes[0,0].text(i - width/2, l + 0.02, f"{l:.3f}", ha="center", fontsize=9)
    axes[0,0].text(i + width/2, i_v + 0.02, f"{i_v:.3f}", ha="center", fontsize=9)

# Plot 2: Score distribution
axes[0,1].hist(iso_anomaly_score[y_test==0], bins=50, alpha=0.6, label="Clean", color="#2ecc71", density=True)
axes[0,1].hist(iso_anomaly_score[y_test==1], bins=50, alpha=0.6, label="AML", color="#e74c3c", density=True)
axes[0,1].set_title("Isolation Forest Anomaly Score Distribution", fontweight="bold")
axes[0,1].set_xlabel("Anomaly Score (higher = more suspicious)"); axes[0,1].legend()

# Plot 3: ROC curves overlaid
from sklearn.metrics import roc_curve
fpr_lgb, tpr_lgb, _ = roc_curve(y_test, y_prob_final)
fpr_iso, tpr_iso, _ = roc_curve(y_test, iso_anomaly_score)
axes[1,0].plot(fpr_lgb, tpr_lgb, "b-", lw=2, label=f"LightGBM (AUC={lgb_auc:.4f})")
axes[1,0].plot(fpr_iso, tpr_iso, "orange", lw=2, label=f"Isolation Forest (AUC={iso_auc:.4f})")
axes[1,0].plot([0,1],[0,1],"k--",alpha=0.3)
axes[1,0].set_title("ROC Curves Comparison", fontweight="bold")
axes[1,0].set_xlabel("False Positive Rate"); axes[1,0].set_ylabel("True Positive Rate")
axes[1,0].legend()

# Plot 4: Agreement matrix
agreement_matrix = np.array([
    [agree_both_clean, disagree_iso_only],
    [disagree_lgb_only, agree_both_aml]
])
sns.heatmap(agreement_matrix, annot=True, fmt=",", cmap="YlOrRd",
            xticklabels=["IsoF: Clean", "IsoF: AML"],
            yticklabels=["LGB: Clean", "LGB: AML"], ax=axes[1,1])
axes[1,1].set_title("LightGBM vs Isolation Forest Agreement Matrix", fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "08_isolation_forest_comparison.png"), bbox_inches="tight")
plt.show()
print(f"  Saved: {OUTPUT_DIR}/08_isolation_forest_comparison.png")

# ══════════════════════════════════════════════════════════════
# STEP 9: Save Predictions & Model
# ══════════════════════════════════════════════════════════════
print("\n── Step 9: Save Outputs ──")

# Save predictions for downstream use
iso_predictions_df = pd.DataFrame({
    "iso_anomaly_score": iso_anomaly_score,
    "iso_score_normalized": iso_score_normalized,
    "iso_predicted_aml": iso_pred,
    "lgb_predicted_aml": y_pred_final,
    "lgb_probability": y_prob_final,
    "actual_is_aml": y_test.values,
    "agreement": ((iso_pred == y_pred_final).astype(int))
})
iso_predictions_df.to_parquet(os.path.join(OUTPUT_DIR, "model_predictions_comparison.parquet"), index=False)
print(f"  Saved: {OUTPUT_DIR}/model_predictions_comparison.parquet")

# Save model
import pickle
with open(os.path.join(OUTPUT_DIR, "isolation_forest_model.pkl"), "wb") as f:
    pickle.dump(iso_model, f)
print(f"  Saved: {OUTPUT_DIR}/isolation_forest_model.pkl")

# Update metadata with Isolation Forest results
import json as _json
metadata_path = os.path.join(OUTPUT_DIR, "model_metadata.json")
if os.path.exists(metadata_path):
    with open(metadata_path) as f:
        metadata = _json.load(f)
else:
    metadata = {}

metadata["isolation_forest"] = {
    "auc_roc": float(iso_auc),
    "f1_score": float(iso_f1),
    "precision": float(iso_prec),
    "recall": float(iso_rec),
    "contamination_rate": float(contamination_rate),
    "n_estimators": 200,
    "agreement_with_lightgbm_pct": float(agreement_pct),
    "both_flagged_aml": int(agree_both_aml),
    "iso_only_flagged": int(disagree_iso_only),
    "lgb_only_flagged": int(disagree_lgb_only)
}
with open(metadata_path, "w") as f:
    _json.dump(metadata, f, indent=2)
print(f"  Updated: {metadata_path}")

print(f"\n{'='*90}")
print("ISOLATION FOREST EVALUATION COMPLETE")
print(f"{'='*90}")
print(f"\n  KEY INSIGHTS:")
print(f"    • LightGBM (supervised) F1:        {lgb_f1:.4f}")
print(f"    • Isolation Forest (unsupervised): {iso_f1:.4f}")
print(f"    • When BOTH flag AML: {both_precision:.1f}% precision (highest confidence)")
print(f"    • When models DISAGREE: investigate for new patterns or model weaknesses")



ISOLATION FOREST — Unsupervised Anomaly Detection

── Step 1: Model Configuration ──
  Contamination rate: 0.3883 (38.8% expected anomalies)
  Number of trees: 200
  Max samples per tree: 256 (default)
  Random state: 42

── Step 2: Training Isolation Forest ──
  Note: This is UNSUPERVISED — y_train is NOT used during training
  The model learns what 'normal' looks like and flags outliers
  Training time: 4.5s

── Step 3: Generate Predictions ──
  Predicted anomalies: 25,980 (38.9% of test set)
  Score range: [0.3378, 0.6032]

── Step 4: Evaluation Against Ground Truth ──

  Isolation Forest Performance:
    AUC-ROC:    0.4928
    F1:         0.3846
    Precision:  0.3842
    Recall:     0.3850
    TP=9,982  FP=15,998  FN=15,947  TN=24,848

── Step 5: Isolation Forest vs LightGBM Head-to-Head ──

  Metric               │     LightGBM   Isolation Forest     Winner
  ──────────────────────────────────────────────────────────────────────
  AUC-ROC              │       0.8659             0

In [14]:
iso_predictions_df['iso_score_percentile'] = pd.Series(iso_predictions_df['iso_anomaly_score']).rank(pct=True).values

In [15]:
iso_predictions_df

,iso_anomaly_score,iso_score_normalized,iso_predicted_aml,lgb_predicted_aml,lgb_probability,actual_is_aml,agreement,iso_score_percentile
0,0.365973,0.106026,0,0,0.000729,0,1,0.232003
1,0.360124,0.083991,0,0,0.198929,0,1,0.142883
2,0.359709,0.082426,0,0,0.363017,0,1,0.136668
3,0.357669,0.074741,0,0,0.422441,0,1,0.108364
4,0.355461,0.066422,0,0,0.438747,0,1,0.080270
...,...,...,...,...,...,...,...,...
66770,0.408350,0.265697,1,1,0.733943,1,1,0.728641
66771,0.392874,0.207383,0,0,0.325897,1,1,0.582853
66772,0.408891,0.267733,1,0,0.375252,0,0,0.732729
66773,0.395475,0.217186,0,0,0.000359,0,1,0.610228


In [16]:
cm_lgb = confusion_matrix(iso_predictions_df['actual_is_aml'], iso_predictions_df['lgb_predicted_aml'])
cm_lgb

array([[31194,  9652],
       [ 5261, 20668]])

In [ ]:
iso_predictions_df['combined_score'] = 0.8 * iso_predictions_df['lgb_probability'] + 0.2 * iso_predictions_df['iso_score_percentile']
iso_predictions_df['combined_pred'] = (iso_predictions_df['combined_score'] >= 0.48).astype(int)

In [19]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

y_true = iso_predictions_df['actual_is_aml'].values
scores = iso_predictions_df['combined_score'].values

thresholds = np.linspace(0.0, 1.0, 101)

rows = []
for t in thresholds:
    y_pred = (scores >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    })

import pandas as pd
df_thr = pd.DataFrame(rows)

# pick what you want to optimize:
best_f1_t = df_thr.loc[df_thr['f1'].idxmax(), 'threshold']

valid = df_thr[df_thr['recall'] >= 0.8]
best_recall_t = valid.loc[valid['threshold'].idxmax(), 'threshold']

#best_recall_t = df_thr.loc[df_thr[df_thr['recall']>=0.8].idxmax(), 'threshold']

valid = df_thr[df_thr['precision'] >= 0.80]
best_prec_t = valid.loc[valid['threshold'].idxmax(), 'threshold']

#best_prec_t = df_thr.loc[df_thr[df_thr['precision']>=0.80].idxmax(), 'threshold']

print("Best thresholds:")
print("  F1-optimal:      ", best_f1_t)
print("  Recall-optimal:  ", best_recall_t)
print("  Precision-optimal:", best_prec_t)

Best thresholds:
  F1-optimal:       0.48
  Recall-optimal:   0.44
  Precision-optimal: 0.99


In [20]:
print("Precision Combined", precision_score(iso_predictions_df['actual_is_aml'], iso_predictions_df['combined_pred']))
print("Recall Combined", recall_score(iso_predictions_df['actual_is_aml'], iso_predictions_df['combined_pred']))

Precision Combined 0.6948070175438597
Recall Combined 0.7637008754676231


In [21]:
print("Precision LGBM", precision_score(iso_predictions_df['actual_is_aml'], iso_predictions_df['lgb_predicted_aml']))
print("Recall LGBM", recall_score(iso_predictions_df['actual_is_aml'], iso_predictions_df['lgb_predicted_aml']))

Precision LGBM 0.6816622691292876
Recall LGBM 0.7970997724555517


In [22]:
print("Precision Isolation Forest", precision_score(iso_predictions_df['actual_is_aml'], iso_predictions_df['iso_predicted_aml']))
print("Recall Isolation Forest", recall_score(iso_predictions_df['actual_is_aml'], iso_predictions_df['iso_predicted_aml']))

Precision Isolation Forest 0.3842186297151655
Recall Isolation Forest 0.3849743530409966


## 13 — Phase 2: Typology Classification + Business Explanations
- Multiclass LightGBM with 6 hyperparameter configs
- Stratified split ensuring ALL typologies in train AND test
- Elaborate business explanations: typology description + specific rule breaches in plain English
- All model parameters saved to `model_parameters_full.json` for Excel summary


In [ ]:
print("=" * 90)
print("PHASE 2: TYPOLOGY CLASSIFICATION + BUSINESS EXPLANATIONS")
print("=" * 90)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# ═══ STEP 1: Phase 1 score + AML-only data ═══
print("\n── Step 1: Prepare Data ──")
X_full_p1 = df_ml[features_final].copy()
for c in X_full_p1.columns: X_full_p1[c] = pd.to_numeric(X_full_p1[c], errors="coerce").fillna(0)
df_ml["_phase1_score"] = final_model.predict(X_full_p1)
phase2_features = features_final + ["_phase1_score"]

aml_labeled = df_ml[(df_ml["is_aml"]==1) & (df_ml["_primary_typology"]!="") & (df_ml["_primary_typology"]!="Clean")].copy()
typology_classes = sorted(aml_labeled["_primary_typology"].unique())
typ_to_idx = {t:i for i,t in enumerate(typology_classes)}
idx_to_typ = {i:t for t,i in typ_to_idx.items()}
aml_labeled["_typ_label"] = aml_labeled["_primary_typology"].map(typ_to_idx)
n_classes = len(typology_classes)

print(f"  AML transactions: {len(aml_labeled):,} | Classes: {n_classes}")
for t,i in typ_to_idx.items():
    print(f"    {i}: {t:<40s} {(aml_labeled['_typ_label']==i).sum():>8,}")

# ═══ STEP 2: Stratified split (ALL typologies in both) ═══
X_aml = aml_labeled[phase2_features].copy()
for c in X_aml.columns: X_aml[c] = pd.to_numeric(X_aml[c], errors="coerce").fillna(0)
y_aml = aml_labeled["_typ_label"].astype(int)

X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_aml, y_aml, test_size=0.20, random_state=42, stratify=y_aml)

total_p2 = len(y_train_2)
class_weights_p2 = {i: total_p2/(n_classes*max((y_train_2==i).sum(),1)) for i in range(n_classes)}
sample_weights_p2 = np.array([class_weights_p2[y] for y in y_train_2])

print(f"\n  Train: {len(X_train_2):,} | Test: {len(X_test_2):,}")
print(f"  {'Typology':<40s} {'Train':>7s} {'Test':>7s}")
print(f"  {'─'*58}")
for i,t in idx_to_typ.items():
    print(f"  {t:<40s} {(y_train_2==i).sum():>7,} {(y_test_2==i).sum():>7,}")

# ═══ STEP 3: Phase 2 Hyperparameter Tuning ═══
print(f"\n── Step 3: Phase 2 Tuning (6 configs) ──\n")
p2_cfgs = [
    {"name":"Baseline",      "nl":63, "md":8, "mc":20, "lr":0.05,"ra":0,  "rl":0},
    {"name":"Deep+Reg",      "nl":127,"md":10,"mc":15, "lr":0.03,"ra":0.1,"rl":1.0},
    {"name":"Slow+Deep",     "nl":127,"md":12,"mc":10, "lr":0.01,"ra":0.1,"rl":0.5},
    {"name":"Wide+Shallow",  "nl":255,"md":6, "mc":30, "lr":0.03,"ra":0.1,"rl":0.5},
    {"name":"ExtraDeep",     "nl":200,"md":14,"mc":10, "lr":0.01,"ra":0.05,"rl":0.5},
    {"name":"HeavyReg",      "nl":63, "md":8, "mc":50, "lr":0.03,"ra":0.5,"rl":5.0},
]

print(f"  {'Config':<20s} │ {'Acc':>7s} {'MacF1':>7s} {'WtdF1':>7s} │ {'Rounds':>6s}")
print("  " + "─" * 55)
p2_results = {}; best_p2_score = 0
for cfg in p2_cfgs:
    params = {"objective":"multiclass","num_class":n_classes,"metric":"multi_logloss",
              "num_leaves":cfg["nl"],"max_depth":cfg["md"],"min_child_samples":cfg["mc"],
              "learning_rate":cfg["lr"],"reg_alpha":cfg["ra"],"reg_lambda":cfg["rl"],
              "subsample":0.8,"colsample_bytree":0.8,"verbosity":-1,"random_state":42,"n_jobs":-1}
    ds2=lgb.Dataset(X_train_2,label=y_train_2,weight=sample_weights_p2)
    val2=lgb.Dataset(X_test_2,label=y_test_2,reference=ds2)
    mdl=lgb.train(params,ds2,num_boost_round=1000,valid_sets=[val2],callbacks=[lgb.early_stopping(50),lgb.log_evaluation(0)])
    proba=mdl.predict(X_test_2); pred=proba.argmax(axis=1)
    acc=accuracy_score(y_test_2,pred); mf1=f1_score(y_test_2,pred,average="macro"); wf1=f1_score(y_test_2,pred,average="weighted")
    p2_results[cfg["name"]]={"acc":acc,"mf1":mf1,"wf1":wf1,"model":mdl,"proba":proba,"pred":pred,
                              "config":cfg,"rounds":mdl.best_iteration if hasattr(mdl,"best_iteration") else 0}
    comp=0.5*acc+0.5*mf1; marker=""
    if comp>best_p2_score: best_p2_score=comp; best_p2=cfg["name"]; marker=" ◄"
    print(f"  {cfg['name']:<20s} │ {acc:>6.4f} {mf1:>6.4f} {wf1:>6.4f} │ {p2_results[cfg['name']]['rounds']:>6}{marker}")

model_p2=p2_results[best_p2]["model"]; y_proba_2=p2_results[best_p2]["proba"]
y_pred_2=p2_results[best_p2]["pred"]; accuracy=p2_results[best_p2]["acc"]
print(f"\n  ► Best: {best_p2} (Accuracy={accuracy:.4f})")

# ═══ STEP 4: Evaluation ═══
print(f"\n── Step 4: Phase 2 Evaluation ──")
print(classification_report(y_test_2, y_pred_2, target_names=typology_classes, digits=4))

# ═══ STEP 5: Build Output Tables with ELABORATE Business Explanations ═══
print(f"\n── Step 5: Build Production Output ──")

fraud_risk_score = df_ml["_phase1_score"]*100
predicted_aml = (df_ml["_phase1_score"]>=best_thresh).astype(int)
rule_cols_list = [c for c in df_ml.columns if c.startswith("rule_") and c not in {"rule_score","rules_triggered","rules_triggered_count"}]
rule_trigger_count = df_ml[rule_cols_list].sum(axis=1).astype(int)
rule_score_vals = pd.to_numeric(df_ml.get("rule_score",0),errors="coerce").fillna(0)

txn_id_col=next((c for c in ["transaction_id"] if c in df.columns),None)
cif_col=next((c for c in ["customer_cif_id","customer_account_number"] if c in df.columns),None)
amt_col=next((c for c in ["transaction_amount"] if c in df.columns),None)
channel_col=next((c for c in ["transaction_mode_channel_bank","transaction_mode_channel_ppi"] if c in df.columns),None)

has_rules=rule_trigger_count>0; has_ml=predicted_aml==1
alert_source=np.select([has_rules&has_ml,has_rules&~has_ml,~has_rules&has_ml,~has_rules&~has_ml],
                        ["Rule + ML Confirmed","Rule Triggered","ML Behavioural Alert","Normal"],default="Normal")

phase1_output = pd.DataFrame({
    "transaction_id": df[txn_id_col].values if txn_id_col else range(len(df)),
    "customer_id": df[cif_col].values if cif_col else "",
    "amount": df[amt_col].values if amt_col else 0,
    "channel": df[channel_col].values if channel_col else "",
    "fraud_risk_score": np.round(fraud_risk_score.values,1),
    "rule_trigger_count": rule_trigger_count.values,
    "weighted_rule_score": rule_score_vals.values.astype(int),
    "alert_source": alert_source,
})

# ═══ ELABORATE BUSINESS EXPLANATIONS ═══
# Rule name → human-readable explanation mapping
RULE_EXPLANATIONS = {
    "rule_freq_2x_business": "Transaction frequency exceeds 2x the customer's historical baseline for business accounts",
    "rule_freq_2x_individual": "Transaction frequency exceeds 2x the customer's historical baseline",
    "rule_series_credits_7d": "Series of credit transactions in 7 days exceeding risk-adjusted threshold",
    "rule_series_debits_7d": "Series of debit transactions in 7 days exceeding risk-adjusted threshold",
    "rule_repeated_counterparty_7d": "Repeated transactions with the same counterparty within 7 days",
    "rule_off_hours_activity": "Significant transaction activity during non-business hours (midnight to 5 AM)",
    "rule_cross_entity_common": "Cross-entity transactions linked to a common person or beneficial owner",
    "rule_integrated_cash_8_5_10L": "Cumulative cash transactions in 30 days approaching RBI reporting threshold (₹8.5L-10L)",
    "rule_structuring_pattern": "Transaction amounts appear structured just below regulatory reporting thresholds (possible smurfing)",
    "rule_multiple_parties_cash": "Cash deposits received from 3 or more different depositors in 7 days",
    "rule_large_cr_dr_business": "Large credit followed by proportional debit within 30 days — possible layering",
    "rule_credit_then_cash": "Credit received then immediately converted to cash withdrawal (50%+ of credit amount)",
    "rule_dormant_activation": "Previously dormant account reactivated with high-value transaction",
    "rule_negative_list_country": "Transaction involves counterparty in FATF high-risk or sanctioned jurisdiction",
    "rule_tax_haven_remit": "Remittance to known tax haven jurisdiction (Switzerland, Cayman Islands, Panama, etc.)",
    "rule_large_cash_business": "Large cash transaction by business entity exceeding risk-adjusted dynamic threshold",
    "rule_large_cash_individual": "Large cash transaction by individual exceeding risk-adjusted dynamic threshold",
    "rule_pep_large_any": "Politically Exposed Person (PEP) involved in large transaction exceeding risk threshold",
    "rule_new_indiv_cash_30pct": "New account (<6 months) with cash transaction exceeding 30% of declared annual income",
    "rule_dormant_75pct_drain": "Dormant account reactivated with immediate withdrawal of 75%+ of balance",
    "rule_multi_channel_24h": "Transaction activity across 4+ different channels within 24 hours — possible account testing",
    "rule_vpn_emulator_detected": "VPN, proxy, or device emulator detected — digital identity masking attempt",
    "rule_device_hopping": "3+ different devices used within 2 hours — possible credential sharing or theft",
    "rule_new_device_new_loc": "Transaction from both new device and new geographic location simultaneously",
    "rule_shell_company": "Entity pattern: high transaction count but cumulative value <20% of balance — shell company indicator",
    "rule_rapid_burst": "Burst of 3+ transactions within 1 hour — unusual velocity spike",
    "rule_ppi_small_kyc_load_breach": "Prepaid wallet with minimum KYC received load exceeding ₹10,000 regulatory limit",
    "rule_ppi_multi_wallet_kyc": "Same PAN/Aadhaar linked to 3+ prepaid wallets with high aggregate activity",
    "rule_ppi_shared_ip_cluster": "10+ different wallets transacting from same IP address — coordinated network indicator",
    "rule_ppi_high_risk_mcc": "Over 70% of wallet spend directed to high-risk merchant categories (gambling, crypto, gaming)",
    "rule_ppi_w2w_layering": "Wallet-to-wallet transfers across 3+ hops within 24 hours — layering pattern",
    "rule_impossible_travel": "Transaction location implies physically impossible travel speed between consecutive transactions",
    "rule_ppi_negative_list_device": "Device, IP address, or UPI VPA found on institutional negative/watchlist",
}

TYPOLOGY_EXPLANATIONS = {
    "Charity Abuse": "This transaction is part of a pattern where funds are being channeled through trust/society/NGO accounts. High-value donations or transfers to charitable entities that show suspicious patterns — such as immediate re-routing of donated funds, transfers to high-risk jurisdictions, or concentration of donations from related parties — indicate potential misuse of charitable vehicles for money laundering.",
    "Circular Transaction Loop": "This transaction participates in a circular flow where funds leave an account and return to the same account (or a connected account) through a chain of intermediaries. The circular pattern is designed to create an appearance of legitimate business activity while the money effectively returns to its origin with a fabricated audit trail.",
    "Funnel Account Network": "This transaction is part of a funnel pattern where multiple small deposits or transfers from different sources converge into a single recipient account. The many-to-one flow pattern suggests the receiving account is acting as a collection point, which is a common money laundering technique to aggregate illicit funds from distributed sources.",
    "High-Risk Corridor Transfer": "This transaction involves a cross-border transfer through a high-risk financial corridor. The counterparty jurisdiction is classified as high-risk by FATF, and the transaction amount significantly exceeds the customer's typical cross-border activity. This combination of geographic risk and behavioral anomaly warrants enhanced due diligence.",
    "Money Mule Network": "This account shows patterns consistent with a money mule — receiving funds and rapidly forwarding them to other accounts. The account holder's profile (age, income, occupation) does not match the volume and velocity of transactions, suggesting the account is being used by a third party to move illicit funds.",
    "Pass-Through Transit Hub": "This account is acting as a transit point where received funds are immediately re-forwarded to other accounts with minimal holding time. The ratio of inflows to outflows approaches 1:1, and the account maintains near-zero end-of-day balance — classic indicators of a pass-through arrangement designed to obscure the source of funds.",
    "Rapid Multi-Hop Layering": "This transaction is part of a rapid layering sequence where funds are moved through multiple accounts in quick succession. The speed and number of hops between accounts are designed to make it difficult for investigators to trace the original source of funds. Each hop typically involves different channels, amounts, or jurisdictions.",
    "Structuring (Smurfing)": "This transaction is part of a structuring pattern where amounts are deliberately kept below regulatory reporting thresholds. Multiple transactions just under the CTR threshold (₹10 lakh for cash, ₹50,000 for certain electronic transfers) from the same customer or related parties indicate intentional avoidance of mandatory reporting requirements.",
    "Third-Party Payment Web": "This transaction is part of a complex web of transfers involving multiple third-party intermediary accounts. The network of transactions creates a web-like pattern where funds are distributed and recombined through several unrelated accounts, making it difficult to identify the true beneficial owner or the ultimate purpose of the funds.",
}

def build_elaborate_explanation(row_pos, typology, confidence, source, score, triggered_rules_list):
    parts = []
    
    # Primary typology explanation
    if typology and typology != "None" and typology in TYPOLOGY_EXPLANATIONS:
        parts.append(f"TYPOLOGY DETECTED ({confidence*100:.0f}% confidence): {typology}")
        parts.append(TYPOLOGY_EXPLANATIONS[typology])
    
    # Rule breach explanations
    if triggered_rules_list:
        rule_explanations = []
        for rule_name in triggered_rules_list[:5]:  # Top 5 rules
            if rule_name in RULE_EXPLANATIONS:
                rule_explanations.append(f"  - {RULE_EXPLANATIONS[rule_name]}")
            else:
                readable = rule_name.replace("rule_","").replace("_"," ").title()
                rule_explanations.append(f"  - {readable} threshold breached")
        if rule_explanations:
            parts.append(f"RULES BREACHED ({len(triggered_rules_list)} rules triggered, weighted score={int(score)}):")
            parts.extend(rule_explanations)
    
    # Alert source context
    if source == "Rule + ML Confirmed":
        parts.append("CONFIDENCE: HIGH — Both regulatory rules AND ML behavioural model independently flag this transaction.")
    elif source == "ML Behavioural Alert":
        parts.append("CONFIDENCE: MODERATE — ML model detects behavioural anomaly not captured by existing rules. Recommend manual review.")
    elif source == "Rule Triggered":
        parts.append("CONFIDENCE: STANDARD — Regulatory rule triggered. No ML behavioural anomaly detected. Routine compliance check.")
    
    if not parts:
        return "Transaction consistent with historical customer behaviour. No rules triggered, no ML anomaly detected."
    
    return " | ".join(parts)

# Build Phase 2 outputs
pred_aml_mask = predicted_aml == 1
X_for_p2 = df_ml.loc[pred_aml_mask, phase2_features].copy()
for c in X_for_p2.columns: X_for_p2[c] = pd.to_numeric(X_for_p2[c], errors="coerce").fillna(0)

p2_typology=["None"]*len(df_ml); p2_confidence=np.zeros(len(df_ml))
p2_priority=["Low"]*len(df_ml); p2_explanation=[""]*len(df_ml)
p2_prob_data={i:np.zeros(len(df_ml)) for i in range(n_classes)}

if len(X_for_p2) > 0:
    proba_full = model_p2.predict(X_for_p2)
    pred_positions = np.where(pred_aml_mask.values)[0]
    
    for j, pos in enumerate(pred_positions):
        typ_idx=proba_full[j].argmax(); typ_name=idx_to_typ[typ_idx]; conf=float(proba_full[j].max())
        src=alert_source[pos]; score=fraud_risk_score.iloc[pos]
        
        p2_typology[pos]=typ_name; p2_confidence[pos]=conf
        if src=="Rule + ML Confirmed" and conf>=0.50: p2_priority[pos]="Critical"
        elif score>=70 or conf>=0.60: p2_priority[pos]="High"
        elif score>=40 or conf>=0.35 or src=="Rule Triggered": p2_priority[pos]="Medium"
        
        # Get triggered rules for this transaction
        triggered = [r for r in rule_cols_list if r in df_ml.columns and df_ml.iloc[pos].get(r,0)==1]
        p2_explanation[pos] = build_elaborate_explanation(pos, typ_name, conf, src, score, triggered)
        for k in range(n_classes): p2_prob_data[k][pos]=float(proba_full[j,k])

# Non-AML explanations
for pos in range(len(df_ml)):
    if not pred_aml_mask.iloc[pos]:
        src=alert_source[pos]; score=rule_score_vals.iloc[pos]
        if src=="Rule Triggered":
            p2_priority[pos]="Medium"
            triggered=[r for r in rule_cols_list if r in df_ml.columns and df_ml.iloc[pos].get(r,0)==1]
            rule_expls = [RULE_EXPLANATIONS.get(r, r.replace("rule_","").replace("_"," ").title()) for r in triggered[:3]]
            p2_explanation[pos] = f"ROUTINE MONITORING: {len(triggered)} compliance rules triggered (score={int(score)}). " + "; ".join(rule_expls)
        else:
            p2_explanation[pos] = "Transaction consistent with historical customer behaviour. No rules triggered, no ML anomaly detected."

phase2_output = pd.DataFrame({
    "transaction_id": phase1_output["transaction_id"].values,
    "customer_id": phase1_output["customer_id"].values,
    "predicted_typology": p2_typology,
    "typology_confidence": np.round(p2_confidence,4),
    "investigation_priority": p2_priority,
    "business_explanation": p2_explanation,
})
for i,t in idx_to_typ.items():
    col=f"prob_{t.lower().replace(' ','_').replace('-','_').replace('(','').replace(')','')}"
    phase2_output[col]=np.round(p2_prob_data[i],4)

# Sample output
print(f"\n  Phase 1: {len(phase1_output):,} rows | Phase 2: {len(phase2_output):,} rows")
print(f"  Alert Sources: " + " | ".join([f"{s}:{(alert_source==s).sum():,}" for s in ["Rule + ML Confirmed","Rule Triggered","ML Behavioural Alert","Normal"]]))

samples=[]
for st in ["Normal","Rule Triggered","ML Behavioural Alert","Rule + ML Confirmed"]:
    n=2 if st!="Rule + ML Confirmed" else 4
    m=phase1_output["alert_source"]==st
    if m.sum()>0: samples.extend(phase1_output[m].sample(min(n,m.sum()),random_state=42).index.tolist())
samples=samples[:10]

print(f"\n  ── Sample Phase 1 Output ──")
print(f"  {'TXN ID':<16s} {'Amount':>10s} {'FRS':>5s} {'Rules':>5s} {'WRS':>5s} {'Alert Source':<25s}")
print(f"  {'─'*75}")
for idx in samples:
    r=phase1_output.iloc[idx]
    print(f"  {str(r['transaction_id'])[:14]:<16s} {r['amount']:>10,.0f} {r['fraud_risk_score']:>5.0f} {int(r['rule_trigger_count']):>5d} {int(r['weighted_rule_score']):>5d} {str(r['alert_source']):<25s}")

print(f"\n  ── Sample Phase 2 Output (with business explanation) ──")
for idx in samples[:5]:
    r2=phase2_output.iloc[idx]
    print(f"\n  TXN: {r2['transaction_id']} | Priority: {r2['investigation_priority']} | Typology: {r2['predicted_typology']} ({r2['typology_confidence']:.2f})")
    print(f"  Explanation: {str(r2['business_explanation'])[:200]}{'...' if len(str(r2['business_explanation']))>200 else ''}")

# ═══ STEP 6: Save everything ═══
print(f"\n── Step 6: Save All Outputs ──")

phase1_output.to_parquet(os.path.join(OUTPUT_DIR,"phase1_aml_detection.parquet"),index=False)
phase2_output.to_parquet(os.path.join(OUTPUT_DIR,"phase2_typology_classification.parquet"),index=False)
combined=phase1_output.merge(phase2_output,on=["transaction_id","customer_id"],how="left")
combined.to_parquet(os.path.join(OUTPUT_DIR,"combined_aml_output.parquet"),index=False)
combined.to_csv(os.path.join(OUTPUT_DIR,"combined_aml_output.csv"),index=False)
model_p2.save_model(os.path.join(OUTPUT_DIR,"phase2_typology_model.txt"))

# Save model parameters for Excel summary
import json as _json
model_params = {
    "phase1": {
        "best_config": best_strategy,
        "config_details": results[best_strategy]["config"],
        "threshold": best_thresh,
        "auc_roc": results[best_strategy]["auc"],
        "f1_score": results[best_strategy]["f1"],
        "precision": results[best_strategy]["precision"],
        "recall": results[best_strategy]["recall"],
        "tp": int(results[best_strategy]["tp"]),
        "fp": int(results[best_strategy]["fp"]),
        "fn": int(results[best_strategy]["fn"]),
        "tn": int(results[best_strategy]["tn"]),
        "best_iteration": int(results[best_strategy]["rounds"]),
        "n_features": len(features_final),
        "n_rules_selected": len(selected_rules),
        "n_rules_removed": len(removed_rules),
        "n_graph_selected": len(selected_graph_features),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "imbalance_ratio": imbalance_ratio,
        "all_configs": {k:{"auc":v["auc"],"f1":v["f1"],"precision":v["precision"],"recall":v["recall"],"rounds":v["rounds"]} for k,v in results.items()},
    },
    "phase2": {
        "best_config": best_p2,
        "config_details": p2_results[best_p2]["config"],
        "accuracy": accuracy,
        "macro_f1": float(f1_score(y_test_2,y_pred_2,average="macro")),
        "weighted_f1": float(f1_score(y_test_2,y_pred_2,average="weighted")),
        "best_iteration": int(p2_results[best_p2]["rounds"]),
        "n_classes": n_classes,
        "classes": typology_classes,
        "n_features": len(phase2_features),
        "n_train": len(X_train_2),
        "n_test": len(X_test_2),
        "all_configs": {k:{"acc":v["acc"],"mf1":v["mf1"],"wf1":v["wf1"],"rounds":v["rounds"]} for k,v in p2_results.items()},
    },
    "features": {
        "final_features": features_final,
        "selected_rules": selected_rules,
        "removed_rules": removed_rules,
        "selected_graph": selected_graph_features,
        "protected_features": list(PROTECTED_FEATURES),
        "encoded_categoricals": encoded_cols,
    }
}
with open(os.path.join(OUTPUT_DIR,"model_parameters_full.json"),"w") as f:
    _json.dump(model_params,f,indent=2,default=str)

typ_mapping={"typ_to_idx":typ_to_idx,"idx_to_typ":{str(k):v for k,v in idx_to_typ.items()}}
with open(os.path.join(OUTPUT_DIR,"typology_mapping.json"),"w") as f:
    _json.dump(typ_mapping,f,indent=2)

print(f"\n  Saved files:")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    fp=os.path.join(OUTPUT_DIR,fn)
    if not os.path.isdir(fp):
        print(f"    {fn:<50s} {os.path.getsize(fp)/(1024*1024):>8.2f} MB")

print(f"\n{'='*90}")
print("PIPELINE COMPLETE")
print(f"{'='*90}")
print(f"  Phase 1: AUC={results[best_strategy]['auc']:.4f} | F1={results[best_strategy]['f1']:.4f} | Recall={results[best_strategy]['recall']:.4f}")
print(f"  Phase 2: Accuracy={accuracy:.4f} | Config={best_p2}")
print(f"  Rules: {len(selected_rules)} kept / {len(removed_rules)} removed (from 126)")
print(f"  Total features: {len(features_final)}")
print(f"  Model params saved to: model_parameters_full.json")



In [19]:
# import pandas as pd
# x_train_read = pd.read_parquet(r"C:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\ml_outputs\X_train.parquet")
# x_train_read.head(2000).to_excel("Check_transactions.xlsx", index=False)